In [ ]:

import json
import shutil
import subprocess
from pathlib import Path

import tensorflow as tf
from google.colab import drive

# Reproducibility
SEED = 42
tf.keras.utils.set_random_seed(SEED)

# Connect Google Drive
drive.mount("/content/drive")

# Project directory
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Applied_AI_Midterm_SRGAN"
)

folders = [
    "data",
    "checkpoints/srgan",
    "checkpoints/model_A",
    "checkpoints/model_B",
    "figures",
    "results",
    "documentation"
]

for folder in folders:
    (PROJECT_DIR / folder).mkdir(
        parents=True, exist_ok=True
    )

# Save project configuration
config_path = PROJECT_DIR / "project_config.json"

if not config_path.exists():
    config = {
        "seed": SEED,
        "dataset": "Dogs vs. Cats",
        "train_ratio": 0.70,
        "test_ratio": 0.30,
        "hr_size": 128,
        "lr_size": 32,
        "srgan_epochs": 150,
        "checkpoint_interval": 10
    }

    with open(config_path, "w") as f:
        json.dump(config, f, indent=4)

# Check environment
print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")
print("Available GPUs:", gpus)

# Check NVIDIA GPU information
if shutil.which("nvidia-smi"):
    gpu_info = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total",
            "--format=csv,noheader"
        ],
        capture_output=True,
        text=True
    )
    print("GPU information:", gpu_info.stdout.strip())
else:
    print("NVIDIA GPU information not available.")

print("Project directory:", PROJECT_DIR)
print("Project folders created successfully.")

if gpus:
    print("\nSUCCESS: GPU is ready for training!")
else:
    print("\nWARNING: No GPU detected.")
    print("Select a GPU in Colab runtime settings.")


In [ ]:

!pip -q install "kagglehub>=1.0.2"

import kagglehub
from pathlib import Path
from collections import Counter
import json

# Download the public dataset
DATASET_HANDLE = "blourdhuraju/dogs-vs-cats-classification"

dataset_path = Path(
    kagglehub.dataset_download(DATASET_HANDLE)
)

# Collect all labeled images
image_extensions = {".jpg", ".jpeg", ".png"}

image_paths = sorted([
    p for p in dataset_path.rglob("*")
    if p.is_file()
    and p.suffix.lower() in image_extensions
    and p.parent.name.lower() in {"cats", "dogs"}
])

# Check the dataset
class_counts = Counter(
    p.parent.name.lower() for p in image_paths
)

print("Dataset source:", DATASET_HANDLE)
print("Dataset location:", dataset_path)
print("\nDataset overview:")
print("Cats:", class_counts["cats"])
print("Dogs:", class_counts["dogs"])
print("Total images:", len(image_paths))

if not image_paths:
    raise RuntimeError(
        "No labeled images were found. "
        "Inspect the downloaded dataset structure."
    )

if not all(class_counts[c] > 0 for c in ["cats", "dogs"]):
    raise RuntimeError("Both classes are required.")

# Record dataset information
dataset_info = {
    "source": DATASET_HANDLE,
    "total_images": len(image_paths),
    "cats": class_counts["cats"],
    "dogs": class_counts["dogs"],
    "random_seed": 42
}

with open(
    PROJECT_DIR / "documentation" / "dataset_info.json",
    "w"
) as f:
    json.dump(dataset_info, f, indent=4)

print("\nDataset downloaded successfully!")


In [ ]:

import pandas as pd
from sklearn.model_selection import train_test_split

# Create dataset table
records = []

for path in image_paths:
    label = 0 if path.parent.name.lower() == "cats" else 1

    records.append({
        "relative_path": str(path.relative_to(dataset_path)),
        "label": label
    })

df = pd.DataFrame(records)

# 70% development data, 30% held-out test data
train_full_df, test_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["label"]
)

# Reserve 10% of development data for validation
train_df, val_df = train_test_split(
    train_full_df,
    test_size=0.10,
    random_state=SEED,
    stratify=train_full_df["label"]
)

# Save split information
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "validation"
test_df["split"] = "test"

split_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

split_file = PROJECT_DIR / "data" / "dataset_split.csv"
split_df.to_csv(split_file, index=False)

# Display class distribution
summary = pd.crosstab(
    split_df["split"],
    split_df["label"]
)

summary = summary.rename(
    columns={0: "Cats", 1: "Dogs"}
)

summary["Total"] = summary.sum(axis=1)

summary = summary.reindex(
    ["train", "validation", "test"]
)

print("Dataset distribution:")
print(summary.to_string())

print("\nTotal images:", len(split_df))
print("Training + Validation:",
      len(train_df) + len(val_df))
print("Testing:", len(test_df))

print("\nSplit saved to:", split_file)

# Check for overlapping file paths
assert split_df["relative_path"].is_unique, (
    "Duplicate image paths detected!"
)

assert len(split_df) == len(df), (
    "Image count mismatch!"
)

print("\nDataset split completed successfully!")


In [ ]:

import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd

# Image dimensions
HR_SIZE = 128
LR_SIZE = 32

def load_hr_image(path):
    image = tf.io.read_file(str(path))
    image = tf.io.decode_image(
        image, channels=3, expand_animations=False
    )

    # Normalize pixel values to [0, 1]
    image = tf.image.convert_image_dtype(
        image, tf.float32
    )

    # Resize to 128x128
    image = tf.image.resize(
        image, [HR_SIZE, HR_SIZE],
        method="bicubic",
        antialias=True
    )

    return tf.clip_by_value(image, 0.0, 1.0)


def augment_example(image, index):
    # Horizontal flipping
    if index % 2 == 0:
        image = tf.image.flip_left_right(image)

    # Small brightness adjustment
    image = tf.image.stateless_random_brightness(
        image,
        max_delta=0.06,
        seed=[SEED, index]
    )

    # Small contrast adjustment
    image = tf.image.stateless_random_contrast(
        image,
        lower=0.9,
        upper=1.1,
        seed=[SEED + 1, index]
    )

    return tf.clip_by_value(image, 0.0, 1.0)


# Select two examples from each class
examples = pd.concat([
    train_df[train_df["label"] == 0].sample(
        n=2, random_state=SEED
    ),
    train_df[train_df["label"] == 1].sample(
        n=2, random_state=SEED
    )
]).reset_index(drop=True)

fig, axes = plt.subplots(
    len(examples), 4, figsize=(13, 12)
)

column_titles = [
    "Original HR (128x128)",
    "Augmented HR (128x128)",
    "Low Resolution (32x32)",
    "Bicubic Upscaling (128x128)"
]

for i, row in examples.iterrows():

    image_path = dataset_path / row["relative_path"]

    # Original normalized HR image
    hr = load_hr_image(image_path)

    # Augmented image
    augmented = augment_example(hr, i)

    # Downscale to 32x32
    lr = tf.image.resize(
        augmented,
        [LR_SIZE, LR_SIZE],
        method="bicubic",
        antialias=True
    )
    lr = tf.clip_by_value(lr, 0.0, 1.0)

    # Standard upscaling for comparison
    bicubic = tf.image.resize(
        lr,
        [HR_SIZE, HR_SIZE],
        method="bicubic"
    )
    bicubic = tf.clip_by_value(bicubic, 0.0, 1.0)

    images = [hr, augmented, lr, bicubic]

    for j, image in enumerate(images):
        axes[i, j].imshow(image.numpy())
        axes[i, j].axis("off")

        if i == 0:
            axes[i, j].set_title(
                column_titles[j], fontsize=10
            )

    label = "Cat" if row["label"] == 0 else "Dog"

    axes[i, 0].set_ylabel(
        label, fontsize=12, fontweight="bold"
    )

plt.suptitle(
    "Image Preprocessing and Augmentation Examples",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()

# Save figure for GitHub documentation
figure_path = (
    PROJECT_DIR / "figures" /
    "preprocessing_examples.png"
)

plt.savefig(
    figure_path, dpi=200, bbox_inches="tight"
)

plt.show()

print("Figure saved to:", figure_path)
print("HR image size:", hr.shape)
print("LR image size:", lr.shape)
print("Pixel range:", float(tf.reduce_min(hr)),
      "to", float(tf.reduce_max(hr)))


In [ ]:

import tensorflow as tf
import pandas as pd
import hashlib
from pathlib import Path
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split

AUTOTUNE = tf.data.AUTOTUNE
BATCH_SIZE = 32
HR_SIZE = 128

# Load original dataset split
split_file = PROJECT_DIR / "data" / "dataset_split.csv"
split_df = pd.read_csv(split_file)

# Local cache for standardized RGB images
PREPARED_DIR = Path("/content/midterm_rgb128")
PREPARED_DIR.mkdir(parents=True, exist_ok=True)

valid_records = []
invalid_records = []

print("Checking and preparing dataset...")

for i, row in enumerate(split_df.itertuples(index=False)):

    relative_path = row.relative_path
    source_path = dataset_path / relative_path

    # Stable filename for each source image
    image_id = hashlib.sha256(
        relative_path.encode("utf-8")
    ).hexdigest()[:24]

    output_path = PREPARED_DIR / f"{image_id}.jpg"

    try:
        if not output_path.exists():

            with Image.open(source_path) as image:

                # Correct orientation and convert to RGB
                image = ImageOps.exif_transpose(image)
                image = image.convert("RGB")

                # Resize to the required HR resolution
                image = image.resize(
                    (HR_SIZE, HR_SIZE),
                    Image.Resampling.BICUBIC
                )

                image.save(
                    output_path,
                    format="JPEG",
                    quality=95
                )

        valid_records.append({
            "relative_path": relative_path,
            "label": row.label,
            "split": row.split,
            "prepared_path": str(output_path)
        })

    except Exception as error:

        invalid_records.append({
            "relative_path": relative_path,
            "error": str(error)
        })

    if (i + 1) % 2500 == 0:
        print(f"Processed {i + 1}/{len(split_df)} images")

print("\nImage preparation completed.")

prepared_df = pd.DataFrame(valid_records)

# Recreate the 70/30 split only if corrupted images
# had to be excluded
if invalid_records:

    print("Unreadable images:", len(invalid_records))

    clean_df = (
        prepared_df[["relative_path", "label", "prepared_path"]]
        .sort_values("relative_path")
        .reset_index(drop=True)
    )

    development_df, test_part = train_test_split(
        clean_df,
        test_size=0.30,
        random_state=SEED,
        stratify=clean_df["label"]
    )

    train_part, val_part = train_test_split(
        development_df,
        test_size=0.10,
        random_state=SEED,
        stratify=development_df["label"]
    )

    train_part = train_part.copy()
    val_part = val_part.copy()
    test_part = test_part.copy()

    train_part["split"] = "train"
    val_part["split"] = "validation"
    test_part["split"] = "test"

    prepared_df = pd.concat(
        [train_part, val_part, test_part],
        ignore_index=True
    )

    # Record excluded files for reproducibility
    pd.DataFrame(invalid_records).to_csv(
        PROJECT_DIR / "data" / "excluded_images.csv",
        index=False
    )

    # Update the persistent dataset split
    prepared_df[
        ["relative_path", "label", "split"]
    ].to_csv(split_file, index=False)

# Separate datasets
train_df = prepared_df[
    prepared_df["split"] == "train"
].reset_index(drop=True)

val_df = prepared_df[
    prepared_df["split"] == "validation"
].reset_index(drop=True)

test_df = prepared_df[
    prepared_df["split"] == "test"
].reset_index(drop=True)

# Load standardized JPEG images
def decode_and_resize(img_path, label):

    image = tf.io.read_file(img_path)

    image = tf.io.decode_jpeg(
        image,
        channels=3
    )

    image = tf.image.convert_image_dtype(
        image, tf.float32
    )

    image.set_shape([HR_SIZE, HR_SIZE, 3])

    return image, tf.cast(label, tf.float32)


def augment_for_classifier(image, label):

    image = tf.image.random_flip_left_right(image)

    image = tf.image.random_brightness(
        image, max_delta=0.08
    )

    image = tf.image.random_contrast(
        image, lower=0.9, upper=1.1
    )

    image = tf.clip_by_value(image, 0.0, 1.0)

    return image, label


def make_dataset(df, training=False):

    paths = df["prepared_path"].tolist()
    labels = df["label"].tolist()

    ds = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    if training:
        ds = ds.shuffle(
            len(df),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        decode_and_resize,
        num_parallel_calls=AUTOTUNE
    )

    if training:
        ds = ds.map(
            augment_for_classifier,
            num_parallel_calls=AUTOTUNE
        )

    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)

# Verify all three datasets
print("\nFinal dataset distribution:")
print(
    pd.crosstab(
        prepared_df["split"],
        prepared_df["label"]
    )
)

print("\nValid images:", len(prepared_df))
print("Excluded images:", len(invalid_records))

for name, ds in [
    ("Train", train_ds),
    ("Validation", val_ds),
    ("Test", test_ds)
]:
    count = 0
    for batch_images, batch_labels in ds:
        count += int(tf.shape(batch_images)[0])

    print(f"{name}: {count} images verified")

print("\nSUCCESS: All datasets are ready for training!")


In [ ]:
import matplotlib.pyplot as plt

class_names = ["Cat", "Dog"]

images, labels = next(iter(train_ds))

print("Batch image tensor shape:", images.shape)
print("Batch label tensor shape:", labels.shape)
print("Pixel range:",
      float(tf.reduce_min(images)),
      "to",
      float(tf.reduce_max(images)))

plt.figure(figsize=(10, 10))

for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy())
    plt.title(class_names[int(labels[i].numpy())])
    plt.axis("off")

plt.suptitle("Sample Training Batch for Model A", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:

import tensorflow as tf
from tensorflow.keras import layers, Model

# Model A configuration
IMG_SIZE = 128
LEARNING_RATE_A = 0.001

# Load pretrained MobileNetV2
base_model_A = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze pretrained layers
base_model_A.trainable = False

# Input layer
inputs = layers.Input(
    shape=(IMG_SIZE, IMG_SIZE, 3)
)

# MobileNetV2 expects pixel values in [-1, 1]
x = layers.Rescaling(
    scale=2.0,
    offset=-1.0
)(inputs)

# Pretrained feature extraction
x = base_model_A(x, training=False)

# Classification head
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)

# Binary classification output
outputs = layers.Dense(
    1,
    activation="sigmoid"
)(x)

# Build model
model_A = Model(
    inputs=inputs,
    outputs=outputs,
    name="Model_A_MobileNetV2"
)

# Compile model
model_A.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE_A
    ),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.AUC(
            name="auc"
        )
    ]
)

# Display architecture
model_A.summary()

print("\nModel A created successfully!")
print("Backbone:", "MobileNetV2")
print("Pretrained weights:", "ImageNet")
print("Input size:", IMG_SIZE, "x", IMG_SIZE)
print("Backbone frozen:", not base_model_A.trainable)
print("Learning rate:", LEARNING_RATE_A)


In [ ]:

from pathlib import Path
import pandas as pd
import tensorflow as tf

# Training configuration
EPOCHS_A = 10
PATIENCE_A = 3

# Output directories
model_a_dir = PROJECT_DIR / "checkpoints" / "model_A"
model_a_dir.mkdir(parents=True, exist_ok=True)

backup_dir = model_a_dir / "training_backup"
best_path = model_a_dir / "model_A_best.keras"
final_path = model_a_dir / "model_A_final.keras"
log_path = PROJECT_DIR / "results" / "model_A_training.csv"

# Handle an empty log file from an interrupted run
if log_path.exists() and log_path.stat().st_size == 0:
    print("Empty training log detected.")
    log_path.unlink()
    print("Empty log removed. Starting with a clean log.")

# Read previous completed epochs, if available
previous_best = None

if log_path.exists():
    try:
        previous_logs = pd.read_csv(log_path)

        if (
            not previous_logs.empty
            and "val_loss" in previous_logs.columns
        ):
            previous_best = float(
                previous_logs["val_loss"].min()
            )

    except pd.errors.EmptyDataError:
        print("Log contains no readable data.")
        log_path.unlink()

# Check whether training is already completed
if final_path.exists():

    print("Completed Model A found.")
    model_A = tf.keras.models.load_model(final_path)

else:

    callbacks_A = [

        # Save training state after every epoch
        tf.keras.callbacks.BackupAndRestore(
            backup_dir=str(backup_dir),
            save_freq="epoch",
            delete_checkpoint=False
        ),

        # Save the best model
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(best_path),
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            initial_value_threshold=previous_best,
            verbose=1
        ),

        # Record training metrics
        tf.keras.callbacks.CSVLogger(
            filename=str(log_path),
            append=(
                log_path.exists()
                and log_path.stat().st_size > 0
            )
        ),

        # Stop if validation loss stops improving
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE_A,
            restore_best_weights=True,
            verbose=1
        )
    ]

    print("Starting Model A training...")

    history_A = model_A.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_A,
        callbacks=callbacks_A,
        verbose=1
    )

    # Load best model
    if best_path.exists():
        model_A = tf.keras.models.load_model(best_path)

    # Save selected final model
    model_A.save(final_path)

    print("\nModel A training completed!")
    print("Best model:", best_path)
    print("Final model:", final_path)
    print("Training log:", log_path)

# Display recorded results
if log_path.exists() and log_path.stat().st_size > 0:

    training_results_A = pd.read_csv(log_path)

    print("\nTraining history:")
    print(training_results_A.tail(10).to_string(index=False))

print("\nModel A ready for evaluation.")


In [ ]:

import json
import shutil
from pathlib import Path

import pandas as pd
import tensorflow as tf

# Fine-tuning configuration
FINE_TUNE_EPOCHS_A = 20
FINE_TUNE_LR_A = 1e-5
FINE_TUNE_LAYERS_A = 30
PATIENCE_A = 5

# Paths
model_dir = PROJECT_DIR / "checkpoints" / "model_A"
results_dir = PROJECT_DIR / "results"

base_path = model_dir / "model_A_best.keras"
fine_path = model_dir / "model_A_finetuned.keras"
final_path = model_dir / "model_A_final.keras"

backup_path = model_dir / "finetuning_backup"
log_path = results_dir / "model_A_finetuning.csv"
base_log_path = results_dir / "model_A_training.csv"
complete_path = model_dir / "finetuning_complete.json"

if not base_path.exists():
    raise FileNotFoundError(
        "Model A checkpoint missing. Complete Cell 8 first."
    )

if not complete_path.exists():

    # Start from the best pretrained classification model
    model_A = tf.keras.models.load_model(base_path)

    # Find MobileNetV2 backbone
    backbone = next(
        layer for layer in model_A.layers
        if isinstance(layer, tf.keras.Model)
        and "mobilenetv2" in layer.name.lower()
    )

    # Unfreeze only the final layers
    backbone.trainable = True

    for layer in backbone.layers[:-FINE_TUNE_LAYERS_A]:
        layer.trainable = False

    # Keep batch normalization frozen
    for layer in backbone.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    # Recompile with a much lower learning rate
    model_A.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=FINE_TUNE_LR_A
        ),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(
                name="accuracy"
            ),
            tf.keras.metrics.AUC(
                name="auc"
            )
        ]
    )

    print("Starting Model A fine-tuning...")
    print("Maximum epochs:", FINE_TUNE_EPOCHS_A)
    print("Learning rate:", FINE_TUNE_LR_A)
    print("Trainable parameters:",
          sum(
              int(tf.size(w))
              for w in model_A.trainable_weights
          ))

    # Handle empty or interrupted log files
    if log_path.exists() and log_path.stat().st_size == 0:
        log_path.unlink()

    previous_best = None

    if log_path.exists():
        try:
            previous_log = pd.read_csv(log_path)
            if not previous_log.empty:
                previous_best = float(
                    previous_log["val_loss"].min()
                )
        except pd.errors.EmptyDataError:
            log_path.unlink()

    callbacks = [
        tf.keras.callbacks.BackupAndRestore(
            backup_dir=str(backup_path),
            save_freq="epoch",
            delete_checkpoint=False
        ),

        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(fine_path),
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            initial_value_threshold=previous_best,
            verbose=1
        ),

        tf.keras.callbacks.CSVLogger(
            str(log_path),
            append=log_path.exists()
        ),

        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE_A,
            restore_best_weights=True,
            verbose=1
        )
    ]

    history_A_fine = model_A.fit(
        train_ds,
        validation_data=val_ds,
        epochs=FINE_TUNE_EPOCHS_A,
        callbacks=callbacks,
        verbose=1
    )

    # Compare both training stages
    base_results = pd.read_csv(base_log_path)
    fine_results = pd.read_csv(log_path)

    base_best_loss = float(
        base_results["val_loss"].min()
    )

    fine_best_loss = float(
        fine_results["val_loss"].min()
    )

    if fine_best_loss < base_best_loss:
        selected_path = fine_path
        selected_stage = "Fine-tuned"
    else:
        selected_path = base_path
        selected_stage = "Initial transfer learning"

    # Preserve the original and fine-tuned checkpoints
    shutil.copy2(selected_path, final_path)

    # Record completion and selected model
    with open(complete_path, "w") as f:
        json.dump({
            "selected_stage": selected_stage,
            "base_best_val_loss": base_best_loss,
            "finetuned_best_val_loss": fine_best_loss,
            "fine_tuning_epochs_completed": len(fine_results),
            "learning_rate": FINE_TUNE_LR_A
        }, f, indent=4)

else:
    print("Fine-tuning was already completed.")
    with open(complete_path) as f:
        result = json.load(f)
    selected_stage = result["selected_stage"]

# Load selected model
model_A = tf.keras.models.load_model(final_path)

print("\nFine-tuning process completed!")
print("Selected model:", selected_stage)
print("Final checkpoint:", final_path)

fine_results = pd.read_csv(log_path)

print("\nFine-tuning results:")
print(fine_results.tail(10).to_string(index=False))

print("\nModel A is ready for evaluation.")


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load training histories
results_dir = PROJECT_DIR / "results"
figures_dir = PROJECT_DIR / "figures"

base_history = pd.read_csv(
    results_dir / "model_A_training.csv"
)

fine_history = pd.read_csv(
    results_dir / "model_A_finetuning.csv"
)

# Combine training stages
combined = pd.concat(
    [base_history, fine_history],
    ignore_index=True
)

epochs = np.arange(1, len(combined) + 1)
transition = len(base_history) + 0.5

def plot_training_metric(metric, ylabel, filename):

    plt.figure(figsize=(9, 5))

    plt.plot(
        epochs,
        combined[metric],
        label="Training",
        linewidth=2
    )

    plt.plot(
        epochs,
        combined["val_" + metric],
        label="Validation",
        linewidth=2
    )

    plt.axvline(
        transition,
        color="gray",
        linestyle="--",
        label="Start of fine-tuning"
    )

    plt.xlabel("Epoch")
    plt.ylabel(ylabel)

    plt.title(
        f"Model A - Training and Validation {ylabel}"
    )

    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()

    output_path = figures_dir / filename

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print("Saved:", output_path)


# Figure 3 - Accuracy
plot_training_metric(
    "accuracy",
    "Accuracy",
    "model_A_accuracy.png"
)

# Figure 4 - Loss
plot_training_metric(
    "loss",
    "Loss",
    "model_A_loss.png"
)

print("\nTotal completed epochs:", len(combined))
print("Initial training:", len(base_history))
print("Fine-tuning:", len(fine_history))
print("\nModel A training figures completed!")


In [ ]:

import tensorflow as tf
from tensorflow.keras import layers, Model
import json

# SRGAN configuration
LR_SIZE = 32
HR_SIZE = 128
SRGAN_BLOCKS = 8
SRGAN_FILTERS = 64

# Pixel shuffle for 2x upscaling
@tf.keras.utils.register_keras_serializable(
    package="Midterm"
)
class PixelShuffle(layers.Layer):

    def __init__(self, scale=2, **kwargs):
        super().__init__(**kwargs)
        self.scale = scale

    def call(self, inputs):
        return tf.nn.depth_to_space(
            inputs, block_size=self.scale
        )

    def get_config(self):
        config = super().get_config()
        config.update({"scale": self.scale})
        return config


# Residual block
def residual_block(x, filters=64):

    skip = x

    x = layers.Conv2D(
        filters, 3, padding="same"
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.PReLU(shared_axes=[1, 2])(x)

    x = layers.Conv2D(
        filters, 3, padding="same"
    )(x)
    x = layers.BatchNormalization()(x)

    return layers.Add()([skip, x])


# Generator
def build_generator():

    inputs = layers.Input(
        shape=(LR_SIZE, LR_SIZE, 3)
    )

    x = layers.Conv2D(
        SRGAN_FILTERS, 9, padding="same"
    )(inputs)
    x = layers.PReLU(shared_axes=[1, 2])(x)

    long_skip = x

    for _ in range(SRGAN_BLOCKS):
        x = residual_block(x, SRGAN_FILTERS)

    x = layers.Conv2D(
        SRGAN_FILTERS, 3, padding="same"
    )(x)
    x = layers.BatchNormalization()(x)

    x = layers.Add()([x, long_skip])

    # Two 2x upscaling stages: 32 -> 64 -> 128
    for _ in range(2):

        x = layers.Conv2D(
            SRGAN_FILTERS * 4,
            3,
            padding="same"
        )(x)

        x = PixelShuffle(scale=2)(x)
        x = layers.PReLU(shared_axes=[1, 2])(x)

    outputs = layers.Conv2D(
        3,
        9,
        padding="same",
        activation="sigmoid"
    )(x)

    return Model(
        inputs, outputs, name="SRGAN_Generator"
    )


# Discriminator
def build_discriminator():

    inputs = layers.Input(
        shape=(HR_SIZE, HR_SIZE, 3)
    )

    x = inputs

    filter_config = [
        (64, 1), (64, 2),
        (128, 1), (128, 2),
        (256, 1), (256, 2),
        (512, 1), (512, 2)
    ]

    for i, (filters, stride) in enumerate(filter_config):

        x = layers.Conv2D(
            filters,
            3,
            strides=stride,
            padding="same"
        )(x)

        if i > 0:
            x = layers.BatchNormalization()(x)

        x = layers.LeakyReLU(
            negative_slope=0.2
        )(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256)(x)
    x = layers.LeakyReLU(
        negative_slope=0.2
    )(x)

    # Raw logit for adversarial loss
    outputs = layers.Dense(1)(x)

    return Model(
        inputs, outputs, name="SRGAN_Discriminator"
    )


# Create networks
generator = build_generator()
discriminator = build_discriminator()

# Verify output dimensions
sample_lr = tf.random.uniform(
    (1, LR_SIZE, LR_SIZE, 3)
)

sample_sr = generator(
    sample_lr, training=False
)

sample_score = discriminator(
    sample_sr, training=False
)

assert sample_sr.shape == (1, 128, 128, 3)
assert sample_score.shape == (1, 1)

# Save architecture settings
architecture_config = {
    "generator_residual_blocks": SRGAN_BLOCKS,
    "generator_filters": SRGAN_FILTERS,
    "lr_size": LR_SIZE,
    "hr_size": HR_SIZE,
    "upscale_factor": 4,
    "image_range": [0, 1],
    "discriminator_output": "logit"
}

config_path = (
    PROJECT_DIR / "documentation" /
    "srgan_architecture.json"
)

with open(config_path, "w") as f:
    json.dump(architecture_config, f, indent=4)

print("SRGAN models created successfully!")
print()
print("Generator parameters:",
      f"{generator.count_params():,}")
print("Discriminator parameters:",
      f"{discriminator.count_params():,}")
print()
print("Generator input:", sample_lr.shape)
print("Generator output:", sample_sr.shape)
print("Discriminator output:", sample_score.shape)
print()
print("Architecture configuration saved.")


In [ ]:

import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

# SRGAN data configuration
SR_BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

# Load one HR image and create its LR counterpart
def load_sr_pair(image_path, training=False):

    image = tf.io.read_file(image_path)

    hr = tf.io.decode_jpeg(
        image, channels=3
    )

    hr = tf.image.convert_image_dtype(
        hr, tf.float32
    )

    hr = tf.ensure_shape(
        hr, [HR_SIZE, HR_SIZE, 3]
    )

    # Apply augmentation before downscaling
    if training:
        hr = tf.image.random_flip_left_right(hr)

    # Create the low-resolution image
    lr = tf.image.resize(
        hr,
        [LR_SIZE, LR_SIZE],
        method="bicubic",
        antialias=True
    )

    lr = tf.clip_by_value(lr, 0.0, 1.0)

    return lr, hr


# Create the SRGAN dataset
def make_sr_dataset(df, training=False):

    paths = df["prepared_path"].tolist()

    ds = tf.data.Dataset.from_tensor_slices(paths)

    if training:
        ds = ds.shuffle(
            buffer_size=len(paths),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        lambda path: load_sr_pair(
            path, training=training
        ),
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(SR_BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)

    return ds


# Create training and validation datasets
sr_train_ds = make_sr_dataset(
    train_df, training=True
)

sr_val_ds = make_sr_dataset(
    val_df, training=False
)

# Check batch dimensions
lr_batch, hr_batch = next(iter(sr_train_ds))

print("SRGAN training dataset ready!")
print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Batch size:", SR_BATCH_SIZE)

print("\nLR batch shape:", lr_batch.shape)
print("HR batch shape:", hr_batch.shape)

print("\nLR pixel range:",
      float(tf.reduce_min(lr_batch)),
      float(tf.reduce_max(lr_batch)))

print("HR pixel range:",
      float(tf.reduce_min(hr_batch)),
      float(tf.reduce_max(hr_batch)))


# Create reproducible visual examples
examples = pd.concat([
    train_df[train_df["label"] == 0].sample(
        n=2, random_state=SEED
    ),
    train_df[train_df["label"] == 1].sample(
        n=2, random_state=SEED
    )
]).reset_index(drop=True)

fig, axes = plt.subplots(
    4, 3, figsize=(10, 12)
)

titles = [
    "Low Resolution (32x32)",
    "Bicubic Upscaling (128x128)",
    "Original HR (128x128)"
]

for i, row in examples.iterrows():

    lr, hr = load_sr_pair(
        tf.constant(row["prepared_path"]),
        training=False
    )

    bicubic = tf.image.resize(
        lr,
        [HR_SIZE, HR_SIZE],
        method="bicubic"
    )

    bicubic = tf.clip_by_value(
        bicubic, 0.0, 1.0
    )

    images = [lr, bicubic, hr]

    for j, image in enumerate(images):

        axes[i, j].imshow(
            image.numpy(),
            interpolation="nearest" if j == 0 else "none"
        )

        axes[i, j].axis("off")

        if i == 0:
            axes[i, j].set_title(titles[j])

    label = "Cat" if row["label"] == 0 else "Dog"
    axes[i, 0].set_ylabel(
        label, fontweight="bold"
    )

plt.suptitle(
    "SRGAN Training Pairs",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout()

# Save documentation figure
figure_path = (
    PROJECT_DIR / "figures" /
    "srgan_training_pairs.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nFigure saved to:", figure_path)
print("SRGAN data preparation completed!")


In [ ]:

import csv
import time
import tensorflow as tf
from pathlib import Path

# Pretraining settings
PRETRAIN_TARGET_EPOCHS = 5
PRETRAIN_LR = 1e-4

pretrain_dir = PROJECT_DIR / "checkpoints" / "srgan" / "pretraining"
pretrain_dir.mkdir(parents=True, exist_ok=True)

checkpoint_dir = pretrain_dir / "resume_checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

best_generator_path = pretrain_dir / "generator_pretrain_best.keras"
history_path = PROJECT_DIR / "results" / "generator_pretraining.csv"

# Compile generator
pretrain_optimizer = tf.keras.optimizers.Adam(
    learning_rate=PRETRAIN_LR,
    beta_1=0.9
)

generator.compile(
    optimizer=pretrain_optimizer,
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(name="mae")
    ]
)

# Build optimizer state before restoring checkpoints
pretrain_optimizer.build(generator.trainable_variables)

# Track completed epochs
completed_epochs = tf.Variable(
    0, dtype=tf.int64, trainable=False
)

checkpoint = tf.train.Checkpoint(
    generator=generator,
    optimizer=pretrain_optimizer,
    completed_epochs=completed_epochs
)

manager = tf.train.CheckpointManager(
    checkpoint,
    directory=str(checkpoint_dir),
    max_to_keep=3
)

# Resume previous training if available
if manager.latest_checkpoint:
    checkpoint.restore(manager.latest_checkpoint)
    print("Restored checkpoint:", manager.latest_checkpoint)
else:
    print("Starting new generator pretraining.")

start_epoch = int(completed_epochs.numpy())

# Find previous best validation loss
best_val_loss = float("inf")

if history_path.exists() and history_path.stat().st_size > 0:
    with open(history_path, newline="") as f:
        previous_rows = list(csv.DictReader(f))

    for row in previous_rows:
        best_val_loss = min(
            best_val_loss,
            float(row["val_loss"])
        )


class PretrainingCheckpoint(tf.keras.callbacks.Callback):

    def on_epoch_begin(self, epoch, logs=None):
        self.start_time = time.monotonic()

    def on_epoch_end(self, epoch, logs=None):
        global best_val_loss

        epoch_number = epoch + 1
        duration = time.monotonic() - self.start_time

        # Save model and optimizer state
        completed_epochs.assign(epoch_number)

        saved_path = manager.save(
            checkpoint_number=epoch_number
        )

        # Save best generator separately
        val_loss = float(logs["val_loss"])

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            generator.save(str(best_generator_path))
            print("New best generator saved.")

        # Record training results
        row = {
            "epoch": epoch_number,
            "loss": float(logs["loss"]),
            "val_loss": val_loss,
            "mae": float(logs["mae"]),
            "val_mae": float(logs["val_mae"]),
            "duration_seconds": round(duration, 2)
        }

        fieldnames = list(row.keys())
        existing_rows = []

        if history_path.exists() and history_path.stat().st_size > 0:
            with open(history_path, newline="") as f:
                existing_rows = list(csv.DictReader(f))

        # Avoid duplicate epochs when resuming
        existing_rows = [
            r for r in existing_rows
            if int(r["epoch"]) != epoch_number
        ]

        existing_rows.append(row)
        existing_rows.sort(key=lambda r: int(r["epoch"]))

        with open(history_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(existing_rows)

        print(f"\nEpoch {epoch_number} checkpoint saved.")
        print(f"Duration: {duration:.1f} seconds")
        print(f"Validation MSE: {val_loss:.6f}")
        print("Checkpoint:", saved_path)


# Run training
print("\nGenerator pretraining")
print("Completed epochs:", start_epoch)
print("Target epochs:", PRETRAIN_TARGET_EPOCHS)
print("Training batches:", len(sr_train_ds))
print("Validation batches:", len(sr_val_ds))

if start_epoch < PRETRAIN_TARGET_EPOCHS:

    generator.fit(
        sr_train_ds,
        validation_data=sr_val_ds,
        initial_epoch=start_epoch,
        epochs=PRETRAIN_TARGET_EPOCHS,
        callbacks=[PretrainingCheckpoint()],
        verbose=1
    )

else:
    print("Target epoch already completed.")

print("\nTotal completed epochs:", int(completed_epochs.numpy()))
print("Best validation MSE:", best_val_loss)
print("Generator pretraining checkpoint ready.")


In [ ]:

import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd

# Load the best pretrained generator
best_generator_path = (
    PROJECT_DIR / "checkpoints" /
    "srgan/pretraining/generator_pretrain_best.keras"
)

generator_preview = tf.keras.models.load_model(
    str(best_generator_path),
    compile=False
)

print("Best pretrained generator loaded.")

# Evaluate reconstruction quality on validation images
metric_totals = {
    "bicubic_psnr": 0.0,
    "bicubic_ssim": 0.0,
    "generator_psnr": 0.0,
    "generator_ssim": 0.0
}

total_samples = 0

for lr_images, hr_images in sr_val_ds:

    # Bicubic baseline
    bicubic_images = tf.image.resize(
        lr_images,
        [HR_SIZE, HR_SIZE],
        method="bicubic"
    )
    bicubic_images = tf.clip_by_value(
        bicubic_images, 0.0, 1.0
    )

    # Generator prediction
    sr_images = generator_preview(
        lr_images, training=False
    )

    sr_images = tf.clip_by_value(
        sr_images, 0.0, 1.0
    )

    # Calculate PSNR
    metric_totals["bicubic_psnr"] += float(
        tf.reduce_sum(
            tf.image.psnr(
                hr_images, bicubic_images, max_val=1.0
            )
        ).numpy()
    )

    metric_totals["generator_psnr"] += float(
        tf.reduce_sum(
            tf.image.psnr(
                hr_images, sr_images, max_val=1.0
            )
        ).numpy()
    )

    # Calculate SSIM
    metric_totals["bicubic_ssim"] += float(
        tf.reduce_sum(
            tf.image.ssim(
                hr_images, bicubic_images, max_val=1.0
            )
        ).numpy()
    )

    metric_totals["generator_ssim"] += float(
        tf.reduce_sum(
            tf.image.ssim(
                hr_images, sr_images, max_val=1.0
            )
        ).numpy()
    )

    total_samples += int(tf.shape(hr_images)[0])


# Calculate mean metrics
metrics = {
    key: value / total_samples
    for key, value in metric_totals.items()
}

print("\nGenerator Pretraining Evaluation")
print("Validation images:", total_samples)

print("\nBicubic:")
print(f"PSNR: {metrics['bicubic_psnr']:.3f} dB")
print(f"SSIM: {metrics['bicubic_ssim']:.4f}")

print("\nPretrained Generator:")
print(f"PSNR: {metrics['generator_psnr']:.3f} dB")
print(f"SSIM: {metrics['generator_ssim']:.4f}")

# Save numerical results
metrics_df = pd.DataFrame([
    {
        "method": "Bicubic",
        "PSNR": metrics["bicubic_psnr"],
        "SSIM": metrics["bicubic_ssim"]
    },
    {
        "method": "Pretrained Generator",
        "PSNR": metrics["generator_psnr"],
        "SSIM": metrics["generator_ssim"]
    }
])

metrics_file = (
    PROJECT_DIR / "results" /
    "generator_pretraining_metrics.csv"
)

metrics_df.to_csv(metrics_file, index=False)


# Select reproducible validation examples
examples = pd.concat([
    val_df[val_df["label"] == 0].sample(
        n=2, random_state=SEED
    ),
    val_df[val_df["label"] == 1].sample(
        n=2, random_state=SEED
    )
]).reset_index(drop=True)

fig, axes = plt.subplots(
    4, 4, figsize=(14, 12)
)

titles = [
    "Low Resolution",
    "Bicubic Upscaling",
    "Pretrained Generator",
    "Original HR"
]

for i, row in examples.iterrows():

    lr, hr = load_sr_pair(
        tf.constant(row["prepared_path"]),
        training=False
    )

    bicubic = tf.image.resize(
        lr,
        [HR_SIZE, HR_SIZE],
        method="bicubic"
    )
    bicubic = tf.clip_by_value(
        bicubic, 0.0, 1.0
    )

    sr = generator_preview(
        tf.expand_dims(lr, axis=0),
        training=False
    )[0]

    sr = tf.clip_by_value(sr, 0.0, 1.0)

    images = [lr, bicubic, sr, hr]

    for j, image in enumerate(images):

        axes[i, j].imshow(
            image.numpy(),
            interpolation="nearest" if j == 0 else "none"
        )

        axes[i, j].axis("off")

        if i == 0:
            axes[i, j].set_title(
                titles[j], fontsize=11
            )

    label = "Cat" if row["label"] == 0 else "Dog"

    axes[i, 0].set_ylabel(
        label,
        fontsize=12,
        fontweight="bold"
    )

plt.suptitle(
    "Generator Pretraining - Image Comparison",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout()

# Save figure
figure_path = (
    PROJECT_DIR / "figures" /
    "generator_pretraining_comparison.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nFigure saved:", figure_path)
print("Metrics saved:", metrics_file)


In [ ]:

import tensorflow as tf

# Load pretrained VGG19
vgg = tf.keras.applications.VGG19(
    include_top=False,
    weights="imagenet",
    input_shape=(128, 128, 3)
)

vgg.trainable = False

# Extract intermediate image features
FEATURE_LAYER = "block3_conv4"

feature_extractor = tf.keras.Model(
    inputs=vgg.input,
    outputs=vgg.get_layer(FEATURE_LAYER).output,
    name="VGG19_Feature_Extractor"
)

feature_extractor.trainable = False

# Convert images to the format expected by VGG19
def prepare_for_vgg(images):

    images = tf.clip_by_value(
        images, 0.0, 1.0
    )

    images = images * 255.0

    return tf.keras.applications.vgg19.preprocess_input(
        images
    )


# Calculate perceptual feature loss
def perceptual_loss(hr_images, sr_images):

    real_features = feature_extractor(
        prepare_for_vgg(hr_images),
        training=False
    )

    fake_features = feature_extractor(
        prepare_for_vgg(sr_images),
        training=False
    )

    return tf.reduce_mean(
        tf.square(
            fake_features - real_features
        )
    )


# Test loss components on a small validation batch
lr_test, hr_test = next(iter(sr_val_ds))

lr_test = lr_test[:4]
hr_test = hr_test[:4]

sr_test = generator_preview(
    lr_test,
    training=False
)

pixel_mse = tf.reduce_mean(
    tf.square(sr_test - hr_test)
)

feature_mse = perceptual_loss(
    hr_test, sr_test
)

print("VGG19 feature extractor ready!")
print("Feature layer:", FEATURE_LAYER)
print("Feature output shape:",
      feature_extractor.output_shape)

print("\nLoss scale check:")
print("Pixel MSE:", float(pixel_mse))
print("Perceptual MSE:", float(feature_mse))

print("\nTrainable VGG parameters:",
      len(feature_extractor.trainable_weights))

print("\nReady for adversarial training setup.")


In [ ]:

import json
import numpy as np
import tensorflow as tf

# SRGAN loss weights
PIXEL_WEIGHT = 1.0
PERCEPTUAL_WEIGHT = 1e-8
ADVERSARIAL_WEIGHT = 1e-3

# GAN training configuration
GENERATOR_LR = 1e-4
DISCRIMINATOR_LR = 1e-4

GAN_TARGET_EPOCHS = 150
GAN_BATCH_SIZE = SR_BATCH_SIZE
CHECKPOINT_INTERVAL = 10

# Load the best pretrained generator
pretrained_generator_path = (
    PROJECT_DIR / "checkpoints" /
    "srgan/pretraining/generator_pretrain_best.keras"
)

generator = tf.keras.models.load_model(
    str(pretrained_generator_path),
    compile=False
)

# Initialize a fresh discriminator
discriminator = build_discriminator()

# Keep the VGG19 feature extractor frozen
feature_extractor.trainable = False

# Binary cross-entropy with logits
bce_loss = tf.keras.losses.BinaryCrossentropy(
    from_logits=True
)

# Test the loss functions without updating weights
lr_sample, hr_sample = next(iter(sr_val_ds))

lr_sample = lr_sample[:4]
hr_sample = hr_sample[:4]

sr_sample = generator(
    lr_sample,
    training=False
)

# Pixel reconstruction loss
pixel_loss_value = tf.reduce_mean(
    tf.square(hr_sample - sr_sample)
)

# VGG feature loss
perceptual_loss_value = perceptual_loss(
    hr_sample,
    sr_sample
)

# Discriminator predictions
real_logits = discriminator(
    hr_sample,
    training=False
)

fake_logits = discriminator(
    sr_sample,
    training=False
)

# Generator adversarial loss
adv_loss_value = bce_loss(
    tf.ones_like(fake_logits),
    fake_logits
)

# Discriminator loss
d_loss_real = bce_loss(
    tf.ones_like(real_logits),
    real_logits
)

d_loss_fake = bce_loss(
    tf.zeros_like(fake_logits),
    fake_logits
)

discriminator_loss_value = (
    d_loss_real + d_loss_fake
)

# Weighted generator contributions
weighted_pixel = PIXEL_WEIGHT * pixel_loss_value
weighted_perceptual = (
    PERCEPTUAL_WEIGHT * perceptual_loss_value
)
weighted_adversarial = (
    ADVERSARIAL_WEIGHT * adv_loss_value
)

total_generator_loss = (
    weighted_pixel
    + weighted_perceptual
    + weighted_adversarial
)

# Check for invalid numerical values
values_to_check = [
    weighted_pixel,
    weighted_perceptual,
    weighted_adversarial,
    total_generator_loss,
    discriminator_loss_value
]

assert all(
    bool(tf.reduce_all(tf.math.is_finite(v)).numpy())
    for v in values_to_check
), "Non-finite loss detected!"

# Save training configuration
gan_config = {
    "target_adversarial_epochs": GAN_TARGET_EPOCHS,
    "batch_size": GAN_BATCH_SIZE,
    "generator_learning_rate": GENERATOR_LR,
    "discriminator_learning_rate": DISCRIMINATOR_LR,
    "pixel_weight": PIXEL_WEIGHT,
    "perceptual_weight": PERCEPTUAL_WEIGHT,
    "adversarial_weight": ADVERSARIAL_WEIGHT,
    "vgg_feature_layer": FEATURE_LAYER,
    "checkpoint_interval": CHECKPOINT_INTERVAL,
    "pretraining_epochs": 5,
    "random_seed": SEED
}

config_path = (
    PROJECT_DIR / "documentation" /
    "srgan_training_config.json"
)

with open(config_path, "w") as f:
    json.dump(gan_config, f, indent=4)

print("SRGAN loss configuration")
print("-" * 45)

print(
    f"Weighted Pixel Loss:      "
    f"{float(weighted_pixel):.6f}"
)

print(
    f"Weighted Perceptual Loss: "
    f"{float(weighted_perceptual):.6f}"
)

print(
    f"Weighted Adversarial Loss:"
    f" {float(weighted_adversarial):.6f}"
)

print("-" * 45)

print(
    f"Total Generator Loss:     "
    f"{float(total_generator_loss):.6f}"
)

print(
    f"Discriminator Loss:       "
    f"{float(discriminator_loss_value):.6f}"
)

print("\nGenerator parameters:",
      f"{generator.count_params():,}")

print("Discriminator parameters:",
      f"{discriminator.count_params():,}")

print("\nConfiguration saved successfully.")
print("Ready to build the adversarial training loop.")



In [ ]:

import os
import csv
import time
import json
import numpy as np
import tensorflow as tf
from pathlib import Path
from datetime import datetime, timezone

# ============================================================
# SRGAN ADVERSARIAL TRAINING
# ============================================================

TOTAL_GAN_EPOCHS = 150

# Change this value to continue training in later sessions
RUN_TO_EPOCH = 1

CHECKPOINT_EVERY = 10
GRADIENT_CLIPNORM = 1.0

# ------------------------------------------------------------
# 1. Output directories
# ------------------------------------------------------------

gan_dir = PROJECT_DIR / "checkpoints" / "srgan" / "adversarial"

resume_dir = gan_dir / "latest_checkpoints"
archive_dir = gan_dir / "archive_checkpoints"

resume_dir.mkdir(parents=True, exist_ok=True)
archive_dir.mkdir(parents=True, exist_ok=True)

history_path = PROJECT_DIR / "results" / "srgan_training.csv"

# Record the additional training setting
config_path = (
    PROJECT_DIR / "documentation" /
    "srgan_training_config.json"
)

with open(config_path, "r") as f:
    saved_config = json.load(f)

saved_config["gradient_clipnorm"] = GRADIENT_CLIPNORM
saved_config["validation_metrics"] = ["PSNR", "SSIM", "MSE"]

with open(config_path, "w") as f:
    json.dump(saved_config, f, indent=4)

# ------------------------------------------------------------
# 2. Create optimizers
# ------------------------------------------------------------

g_optimizer = tf.keras.optimizers.Adam(
    learning_rate=GENERATOR_LR,
    beta_1=0.9,
    global_clipnorm=GRADIENT_CLIPNORM
)

d_optimizer = tf.keras.optimizers.Adam(
    learning_rate=DISCRIMINATOR_LR,
    beta_1=0.9,
    global_clipnorm=GRADIENT_CLIPNORM
)

# Build optimizer states before restoring
g_optimizer.build(generator.trainable_variables)
d_optimizer.build(discriminator.trainable_variables)

# Completed epoch counter
gan_epoch = tf.Variable(
    0,
    dtype=tf.int64,
    trainable=False
)

# ------------------------------------------------------------
# 3. Full training checkpoint
# ------------------------------------------------------------

gan_checkpoint = tf.train.Checkpoint(
    epoch=gan_epoch,
    generator=generator,
    discriminator=discriminator,
    generator_optimizer=g_optimizer,
    discriminator_optimizer=d_optimizer
)

# Keep last 3 checkpoints
resume_manager = tf.train.CheckpointManager(
    checkpoint=gan_checkpoint,
    directory=str(resume_dir),
    max_to_keep=3
)

# Keep all 10-epoch archive checkpoints
archive_manager = tf.train.CheckpointManager(
    checkpoint=gan_checkpoint,
    directory=str(archive_dir),
    max_to_keep=None
)

if resume_manager.latest_checkpoint:

    restore_status = gan_checkpoint.restore(
        resume_manager.latest_checkpoint
    )

    restore_status.assert_existing_objects_matched()

    print("Checkpoint restored:")
    print(resume_manager.latest_checkpoint)

else:

    print("Starting new adversarial training.")

    # Save epoch-zero state for recovery
    initial_checkpoint = resume_manager.save(
        checkpoint_number=0
    )

    print("Initial checkpoint:", initial_checkpoint)

start_epoch = int(gan_epoch.numpy())

# ------------------------------------------------------------
# 4. Training step
# ------------------------------------------------------------

@tf.function
def gan_train_step(lr_images, hr_images):

    # ----- Train Discriminator -----

    # Generate images without updating the generator
    fake_for_d = tf.stop_gradient(
        generator(lr_images, training=False)
    )

    with tf.GradientTape() as d_tape:

        real_logits = discriminator(
            hr_images, training=True
        )

        fake_logits = discriminator(
            fake_for_d, training=True
        )

        d_real_loss = bce_loss(
            tf.ones_like(real_logits),
            real_logits
        )

        d_fake_loss = bce_loss(
            tf.zeros_like(fake_logits),
            fake_logits
        )

        d_loss = d_real_loss + d_fake_loss

    d_gradients = d_tape.gradient(
        d_loss,
        discriminator.trainable_variables
    )

    d_optimizer.apply_gradients(
        zip(d_gradients, discriminator.trainable_variables)
    )

    # ----- Train Generator -----

    with tf.GradientTape() as g_tape:

        sr_images = generator(
            lr_images, training=True
        )

        # Discriminator is evaluated without BN updates
        sr_logits = discriminator(
            sr_images, training=False
        )

        pixel_loss = tf.reduce_mean(
            tf.square(hr_images - sr_images)
        )

        feature_loss = perceptual_loss(
            hr_images, sr_images
        )

        adversarial_loss = bce_loss(
            tf.ones_like(sr_logits),
            sr_logits
        )

        weighted_pixel = PIXEL_WEIGHT * pixel_loss

        weighted_feature = (
            PERCEPTUAL_WEIGHT * feature_loss
        )

        weighted_adv = (
            ADVERSARIAL_WEIGHT * adversarial_loss
        )

        g_loss = (
            weighted_pixel
            + weighted_feature
            + weighted_adv
        )

    g_gradients = g_tape.gradient(
        g_loss,
        generator.trainable_variables
    )

    g_optimizer.apply_gradients(
        zip(g_gradients, generator.trainable_variables)
    )

    # Stop immediately if losses become non-finite
    tf.debugging.assert_all_finite(
        g_loss, "Invalid generator loss"
    )

    tf.debugging.assert_all_finite(
        d_loss, "Invalid discriminator loss"
    )

    return (
        g_loss,
        d_loss,
        weighted_pixel,
        weighted_feature,
        weighted_adv
    )

# ------------------------------------------------------------
# 5. Validation step
# ------------------------------------------------------------

@tf.function
def gan_validation_step(lr_images, hr_images):

    sr_images = generator(
        lr_images, training=False
    )

    sr_images = tf.clip_by_value(
        sr_images, 0.0, 1.0
    )

    mse = tf.reduce_mean(
        tf.square(hr_images - sr_images),
        axis=[1, 2, 3]
    )

    psnr = tf.image.psnr(
        hr_images, sr_images, max_val=1.0
    )

    ssim = tf.image.ssim(
        hr_images, sr_images, max_val=1.0
    )

    return mse, psnr, ssim

# ------------------------------------------------------------
# 6. Save training history safely
# ------------------------------------------------------------

history_fields = [
    "epoch",
    "g_loss",
    "d_loss",
    "pixel_loss",
    "perceptual_loss",
    "adversarial_loss",
    "val_mse",
    "val_psnr",
    "val_ssim",
    "duration_seconds",
    "timestamp_utc"
]

def save_epoch_history(row):

    previous_rows = []

    if history_path.exists() and history_path.stat().st_size > 0:

        with open(history_path, newline="") as f:
            previous_rows = list(csv.DictReader(f))

    # Replace an epoch entry if it was repeated after a crash
    previous_rows = [
        r for r in previous_rows
        if int(r["epoch"]) != int(row["epoch"])
    ]

    previous_rows.append(row)

    previous_rows.sort(
        key=lambda r: int(r["epoch"])
    )

    temporary_path = history_path.with_suffix(".tmp")

    with open(temporary_path, "w", newline="") as f:

        writer = csv.DictWriter(
            f, fieldnames=history_fields
        )

        writer.writeheader()
        writer.writerows(previous_rows)

    os.replace(temporary_path, history_path)

# ------------------------------------------------------------
# 7. Training loop
# ------------------------------------------------------------

if RUN_TO_EPOCH > TOTAL_GAN_EPOCHS:
    raise ValueError("Target exceeds 150 epochs.")

if RUN_TO_EPOCH <= start_epoch:

    print(
        f"Training already completed through epoch "
        f"{start_epoch}."
    )

else:

    print("\nSRGAN training started")
    print("Starting after epoch:", start_epoch)
    print("Run until epoch:", RUN_TO_EPOCH)
    print("Training batches:", len(sr_train_ds))
    print("Validation batches:", len(sr_val_ds))
    print()

    for epoch in range(start_epoch, RUN_TO_EPOCH):

        epoch_start = time.perf_counter()

        # Epoch training metrics
        g_metric = tf.keras.metrics.Mean()
        d_metric = tf.keras.metrics.Mean()
        pixel_metric = tf.keras.metrics.Mean()
        feature_metric = tf.keras.metrics.Mean()
        adv_metric = tf.keras.metrics.Mean()

        print(
            f"\nEpoch {epoch + 1}/{TOTAL_GAN_EPOCHS}"
        )

        for batch_idx, (lr_images, hr_images) in enumerate(
            sr_train_ds, start=1
        ):

            (
                g_loss,
                d_loss,
                pixel_loss,
                feature_loss,
                adv_loss
            ) = gan_train_step(
                lr_images, hr_images
            )

            g_metric.update_state(g_loss)
            d_metric.update_state(d_loss)
            pixel_metric.update_state(pixel_loss)
            feature_metric.update_state(feature_loss)
            adv_metric.update_state(adv_loss)

            if (
                batch_idx % 100 == 0
                or batch_idx == len(sr_train_ds)
            ):

                print(
                    f"Batch {batch_idx}/{len(sr_train_ds)} | "
                    f"G: {float(g_metric.result()):.5f} | "
                    f"D: {float(d_metric.result()):.5f}"
                )

        # ----- Validation -----

        val_mse_sum = 0.0
        val_psnr_sum = 0.0
        val_ssim_sum = 0.0
        val_count = 0

        for lr_images, hr_images in sr_val_ds:

            mse, psnr, ssim = gan_validation_step(
                lr_images, hr_images
            )

            val_mse_sum += float(tf.reduce_sum(mse))
            val_psnr_sum += float(tf.reduce_sum(psnr))
            val_ssim_sum += float(tf.reduce_sum(ssim))

            val_count += int(tf.shape(lr_images)[0])

        val_mse = val_mse_sum / val_count
        val_psnr = val_psnr_sum / val_count
        val_ssim = val_ssim_sum / val_count

        duration = time.perf_counter() - epoch_start
        current_epoch = epoch + 1

        row = {
            "epoch": current_epoch,
            "g_loss": float(g_metric.result()),
            "d_loss": float(d_metric.result()),
            "pixel_loss": float(pixel_metric.result()),
            "perceptual_loss": float(feature_metric.result()),
            "adversarial_loss": float(adv_metric.result()),
            "val_mse": val_mse,
            "val_psnr": val_psnr,
            "val_ssim": val_ssim,
            "duration_seconds": round(duration, 2),
            "timestamp_utc": datetime.now(
                timezone.utc
            ).isoformat()
        }

        # Write metrics and persist full training state
        save_epoch_history(row)

        gan_epoch.assign(current_epoch)

        latest_path = resume_manager.save(
            checkpoint_number=current_epoch
        )

        print("\nEpoch summary")
        print("-" * 45)

        print(f"Generator Loss:     {row['g_loss']:.6f}")
        print(f"Discriminator Loss: {row['d_loss']:.6f}")
        print(f"Validation MSE:     {val_mse:.6f}")
        print(f"Validation PSNR:    {val_psnr:.3f} dB")
        print(f"Validation SSIM:    {val_ssim:.4f}")
        print(f"Duration:           {duration:.1f} seconds")

        print("Checkpoint:", latest_path)

        # Archive every 10 epochs
        if current_epoch % CHECKPOINT_EVERY == 0:

            archive_path = archive_manager.save(
                checkpoint_number=current_epoch
            )

            print("Archive saved:", archive_path)

        # Rough total-time estimate after first epoch
        if current_epoch == 1:

            estimated_hours = (
                duration * TOTAL_GAN_EPOCHS / 3600
            )

            print(
                f"\nEstimated training time "
                f"for 150 epochs: "
                f"{estimated_hours:.1f} hours"
            )

        print("-" * 45)

print("\nCurrent completed GAN epochs:",
      int(gan_epoch.numpy()))

print("Training history:", history_path)
print("Resume checkpoints:", resume_dir)
print("Archive checkpoints:", archive_dir)


In [ ]:

import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# 1. Restore saved adversarial checkpoint
# --------------------------------------------------

latest_checkpoint = tf.train.latest_checkpoint(
    str(resume_dir)
)

if latest_checkpoint is None:
    raise FileNotFoundError(
        "No adversarial checkpoint found."
    )

gan_eval_generator = build_generator()
gan_eval_discriminator = build_discriminator()

diagnostic_epoch = tf.Variable(
    0, dtype=tf.int64, trainable=False
)

diagnostic_checkpoint = tf.train.Checkpoint(
    epoch=diagnostic_epoch,
    generator=gan_eval_generator,
    discriminator=gan_eval_discriminator
)

status = diagnostic_checkpoint.restore(
    latest_checkpoint
)
status.expect_partial()
status.assert_existing_objects_matched()

current_epoch = int(diagnostic_epoch.numpy())

print("Restored GAN epoch:", current_epoch)
print("Checkpoint:", latest_checkpoint)

# Load pretrained generator for comparison
pretrained_generator = tf.keras.models.load_model(
    str(best_generator_path),
    compile=False
)

# --------------------------------------------------
# 2. Evaluate discriminator predictions
# --------------------------------------------------

real_probabilities = []
fake_probabilities = []

# Use 128 validation images
for lr_images, hr_images in sr_val_ds.take(8):

    sr_images = gan_eval_generator(
        lr_images, training=False
    )

    real_logits = gan_eval_discriminator(
        hr_images, training=False
    )

    fake_logits = gan_eval_discriminator(
        sr_images, training=False
    )

    real_probabilities.append(
        tf.sigmoid(real_logits).numpy().flatten()
    )

    fake_probabilities.append(
        tf.sigmoid(fake_logits).numpy().flatten()
    )

import numpy as np

real_probs = np.concatenate(real_probabilities)
fake_probs = np.concatenate(fake_probabilities)

real_accuracy = np.mean(real_probs >= 0.5)
fake_accuracy = np.mean(fake_probs < 0.5)

print("\nDiscriminator diagnostics")
print("-" * 45)

print("Validation images:", len(real_probs))
print(
    f"Mean probability for real images: "
    f"{real_probs.mean():.4f}"
)
print(
    f"Mean probability for generated images: "
    f"{fake_probs.mean():.4f}"
)

print(f"Real image accuracy: {real_accuracy:.2%}")
print(f"Fake image accuracy: {fake_accuracy:.2%}")

# --------------------------------------------------
# 3. Compare generated images
# --------------------------------------------------

examples = pd.concat([
    val_df[val_df["label"] == 0].sample(
        n=2, random_state=SEED
    ),
    val_df[val_df["label"] == 1].sample(
        n=2, random_state=SEED
    )
]).reset_index(drop=True)

fig, axes = plt.subplots(
    4, 5, figsize=(16, 12)
)

titles = [
    "Low Resolution",
    "Bicubic",
    "Pretrained Generator",
    f"SRGAN Epoch {current_epoch}",
    "Original HR"
]

for i, row in examples.iterrows():

    lr, hr = load_sr_pair(
        tf.constant(row["prepared_path"]),
        training=False
    )

    lr_input = tf.expand_dims(lr, axis=0)

    bicubic = tf.image.resize(
        lr, [HR_SIZE, HR_SIZE],
        method="bicubic"
    )

    pre_sr = pretrained_generator(
        lr_input, training=False
    )[0]

    gan_sr = gan_eval_generator(
        lr_input, training=False
    )[0]

    images = [lr, bicubic, pre_sr, gan_sr, hr]

    for j, image in enumerate(images):

        image = tf.clip_by_value(
            image, 0.0, 1.0
        )

        axes[i, j].imshow(
            image.numpy(),
            interpolation="nearest" if j == 0 else "none"
        )

        axes[i, j].axis("off")

        if i == 0:
            axes[i, j].set_title(
                titles[j], fontsize=11
            )

    label = "Cat" if row["label"] == 0 else "Dog"
    axes[i, 0].set_ylabel(
        label, fontweight="bold"
    )

plt.suptitle(
    f"SRGAN Image Comparison - Epoch {current_epoch}",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout()

figure_path = (
    PROJECT_DIR / "figures" /
    f"srgan_epoch_{current_epoch:03d}_comparison.png"
)

plt.savefig(
    figure_path,
    dpi=250,
    bbox_inches="tight"
)

plt.show()

# --------------------------------------------------
# 4. Save diagnostic results
# --------------------------------------------------

diagnostic_path = (
    PROJECT_DIR / "results" /
    "srgan_discriminator_diagnostics.csv"
)

new_result = pd.DataFrame([{
    "epoch": current_epoch,
    "real_probability": float(real_probs.mean()),
    "fake_probability": float(fake_probs.mean()),
    "real_accuracy": float(real_accuracy),
    "fake_accuracy": float(fake_accuracy)
}])

if diagnostic_path.exists():
    old_results = pd.read_csv(diagnostic_path)
    old_results = old_results[
        old_results["epoch"] != current_epoch
    ]
    new_result = pd.concat(
        [old_results, new_result],
        ignore_index=True
    )

new_result = new_result.sort_values("epoch")
new_result.to_csv(diagnostic_path, index=False)

print("\nFigure saved:", figure_path)
print("Diagnostics saved:", diagnostic_path)


In [ ]:

import numpy as np
import tensorflow as tf

# Use the diagnostic copies from Cell 18
# The saved GAN checkpoint is not modified.

lr_images, hr_images = next(iter(sr_val_ds))

sr_images = gan_eval_generator(
    lr_images, training=False
)

def show_d_results(name, real_scores, fake_scores):
    real = tf.sigmoid(real_scores).numpy().ravel()
    fake = tf.sigmoid(fake_scores).numpy().ravel()

    print(f"\n{name}")
    print("-" * 45)
    print(f"Real probability: {real.mean():.4f}")
    print(f"Fake probability: {fake.mean():.4f}")
    print(f"Real accuracy:    {(real >= 0.5).mean():.2%}")
    print(f"Fake accuracy:    {(fake < 0.5).mean():.2%}")

# 1. Inference mode
real_eval = gan_eval_discriminator(
    hr_images, training=False
)
fake_eval = gan_eval_discriminator(
    sr_images, training=False
)

show_d_results(
    "TEST 1: Inference mode",
    real_eval, fake_eval
)

# 2. Training mode with separate real/fake batches
real_train = gan_eval_discriminator(
    hr_images, training=True
)
fake_train = gan_eval_discriminator(
    sr_images, training=True
)

show_d_results(
    "TEST 2: Training mode - separate batches",
    real_train, fake_train
)

# 3. Training mode with real and fake images mixed
mixed_images = tf.concat(
    [hr_images, sr_images], axis=0
)

mixed_logits = gan_eval_discriminator(
    mixed_images, training=True
)

batch_size = int(tf.shape(hr_images)[0])

show_d_results(
    "TEST 3: Training mode - mixed batch",
    mixed_logits[:batch_size],
    mixed_logits[batch_size:]
)

print("\nDiagnostics completed.")
print("Saved GAN checkpoints remain unchanged.")



In [ ]:

import json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model

# --------------------------------------------------
# 1. Updated Discriminator with Group Normalization
# --------------------------------------------------

def build_discriminator():

    inputs = layers.Input(
        shape=(HR_SIZE, HR_SIZE, 3)
    )

    x = inputs

    filter_config = [
        (64, 1),
        (64, 2),
        (128, 1),
        (128, 2),
        (256, 1),
        (256, 2),
        (512, 1),
        (512, 2)
    ]

    for i, (filters, stride) in enumerate(filter_config):

        x = layers.Conv2D(
            filters,
            kernel_size=3,
            strides=stride,
            padding="same"
        )(x)

        # Group Normalization instead of Batch Normalization
        if i > 0:
            x = layers.GroupNormalization(
                groups=32,
                axis=-1,
                epsilon=1e-5
            )(x)

        x = layers.LeakyReLU(
            negative_slope=0.2
        )(x)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(256)(x)

    x = layers.LeakyReLU(
        negative_slope=0.2
    )(x)

    outputs = layers.Dense(1)(x)

    return Model(
        inputs,
        outputs,
        name="SRGAN_Discriminator_GroupNorm"
    )


# --------------------------------------------------
# 2. Restore the pretrained Generator
# --------------------------------------------------

generator = tf.keras.models.load_model(
    str(best_generator_path),
    compile=False
)

# Initialize the updated Discriminator
tf.keras.utils.set_random_seed(SEED + 101)

discriminator = build_discriminator()

# --------------------------------------------------
# 3. Verify consistent Discriminator behavior
# --------------------------------------------------

lr_images, hr_images = next(iter(sr_val_ds))

lr_images = lr_images[:8]
hr_images = hr_images[:8]

sr_images = generator(
    lr_images,
    training=False
)

# Same images, different execution modes
logits_eval = discriminator(
    hr_images,
    training=False
)

logits_train = discriminator(
    hr_images,
    training=True
)

# Check whether mixing images changes predictions
mixed_images = tf.concat(
    [hr_images, sr_images],
    axis=0
)

mixed_logits = discriminator(
    mixed_images,
    training=True
)

mode_difference = float(
    tf.reduce_max(
        tf.abs(logits_eval - logits_train)
    ).numpy()
)

batch_difference = float(
    tf.reduce_max(
        tf.abs(logits_train - mixed_logits[:8])
    ).numpy()
)

assert mode_difference < 1e-4, (
    "Training and inference predictions differ."
)

assert batch_difference < 1e-4, (
    "Predictions depend on other batch images."
)

# --------------------------------------------------
# 4. Save updated architecture configuration
# --------------------------------------------------

new_config = {
    "architecture_version": "groupnorm_v2",
    "generator": "8 residual blocks, 64 filters",
    "generator_pretraining_epochs": 5,
    "discriminator_normalization": "GroupNormalization",
    "normalization_groups": 32,
    "discriminator_layers": 8,
    "train_real_fake_in_mixed_batch": True,
    "adversarial_training_restart": True,
    "adversarial_target_epochs": 150,
    "seed": SEED + 101
}

config_file = (
    PROJECT_DIR / "documentation" /
    "srgan_architecture_v2.json"
)

with open(config_file, "w") as f:
    json.dump(new_config, f, indent=4)

# --------------------------------------------------
# 5. Display results
# --------------------------------------------------

print("Updated SRGAN architecture ready!")
print("-" * 45)

print("Generator parameters:",
      f"{generator.count_params():,}")

print("Discriminator parameters:",
      f"{discriminator.count_params():,}")

print("\nNormalization consistency check:")
print(f"Train/inference difference: {mode_difference:.8f}")
print(f"Mixed-batch difference:      {batch_difference:.8f}")

print("\nBoth consistency checks passed.")
print("Pretrained generator successfully restored.")
print("New discriminator initialized.")
print("Architecture configuration saved.")



In [ ]:

import os
import csv
import time
import json
import numpy as np
import tensorflow as tf
from pathlib import Path
from datetime import datetime, timezone

# ============================================================
# SRGAN TRAINING V2 - GROUP NORMALIZATION
# ============================================================

TOTAL_GAN_EPOCHS = 150
RUN_TO_EPOCH = 15
CHECKPOINT_INTERVAL = 10

# Verify that the updated architecture is active
assert discriminator.name == "SRGAN_Discriminator_GroupNorm", (
    "Run Cell 20 first to initialize the updated discriminator."
)

feature_extractor.trainable = False

# ------------------------------------------------------------
# 1. Directories and configuration
# ------------------------------------------------------------

gan_v2_dir = (
    PROJECT_DIR / "checkpoints" /
    "srgan" / "groupnorm_v2"
)

resume_dir_v2 = gan_v2_dir / "resume"
archive_dir_v2 = gan_v2_dir / "archive"

resume_dir_v2.mkdir(parents=True, exist_ok=True)
archive_dir_v2.mkdir(parents=True, exist_ok=True)

history_path_v2 = (
    PROJECT_DIR / "results" /
    "srgan_training_groupnorm_v2.csv"
)

if not 1 <= RUN_TO_EPOCH <= TOTAL_GAN_EPOCHS:
    raise ValueError("RUN_TO_EPOCH must be between 1 and 150.")

# ------------------------------------------------------------
# 2. Optimizers
# ------------------------------------------------------------

g_optimizer_v2 = tf.keras.optimizers.Adam(
    learning_rate=GENERATOR_LR,
    beta_1=0.9,
    global_clipnorm=1.0
)

d_optimizer_v2 = tf.keras.optimizers.Adam(
    learning_rate=DISCRIMINATOR_LR,
    beta_1=0.9,
    global_clipnorm=1.0
)

g_optimizer_v2.build(generator.trainable_variables)
d_optimizer_v2.build(discriminator.trainable_variables)

completed_gan_epochs_v2 = tf.Variable(
    0, dtype=tf.int64, trainable=False
)

# ------------------------------------------------------------
# 3. Full checkpoints
# ------------------------------------------------------------

checkpoint_v2 = tf.train.Checkpoint(
    generator=generator,
    discriminator=discriminator,
    generator_optimizer=g_optimizer_v2,
    discriminator_optimizer=d_optimizer_v2,
    epoch=completed_gan_epochs_v2
)

resume_manager_v2 = tf.train.CheckpointManager(
    checkpoint_v2,
    directory=str(resume_dir_v2),
    max_to_keep=3
)

archive_manager_v2 = tf.train.CheckpointManager(
    checkpoint_v2,
    directory=str(archive_dir_v2),
    max_to_keep=None
)

if resume_manager_v2.latest_checkpoint:

    status = checkpoint_v2.restore(
        resume_manager_v2.latest_checkpoint
    )

    status.assert_existing_objects_matched()

    print("Restored checkpoint:")
    print(resume_manager_v2.latest_checkpoint)

else:

    print("Starting new GroupNorm SRGAN training.")

    initial_path = resume_manager_v2.save(
        checkpoint_number=0
    )

    print("Initial checkpoint:", initial_path)

start_epoch_v2 = int(completed_gan_epochs_v2.numpy())

# ------------------------------------------------------------
# 4. GAN training step
# ------------------------------------------------------------

@tf.function
def train_step_v2(lr_images, hr_images):

    # ----- Discriminator update -----

    generated_for_d = tf.stop_gradient(
        generator(lr_images, training=False)
    )

    batch_size = tf.shape(hr_images)[0]

    # Real and fake images are combined into one batch
    combined_images = tf.concat(
        [hr_images, generated_for_d],
        axis=0
    )

    combined_labels = tf.concat([
        tf.ones((batch_size, 1)),
        tf.zeros((batch_size, 1))
    ], axis=0)

    # Shuffle images and corresponding labels together
    indices = tf.random.shuffle(
        tf.range(tf.shape(combined_images)[0])
    )

    combined_images = tf.gather(
        combined_images, indices
    )

    combined_labels = tf.gather(
        combined_labels, indices
    )

    with tf.GradientTape() as d_tape:

        mixed_logits = discriminator(
            combined_images,
            training=True
        )

        d_loss = bce_loss(
            combined_labels,
            mixed_logits
        )

    tf.debugging.assert_all_finite(
        d_loss, "Non-finite discriminator loss"
    )

    d_gradients = d_tape.gradient(
        d_loss,
        discriminator.trainable_variables
    )

    d_optimizer_v2.apply_gradients(
        zip(
            d_gradients,
            discriminator.trainable_variables
        )
    )

    # ----- Generator update -----

    with tf.GradientTape() as g_tape:

        sr_images = generator(
            lr_images, training=True
        )

        fake_logits = discriminator(
            sr_images, training=False
        )

        # Reconstruction loss
        pixel_loss = tf.reduce_mean(
            tf.square(hr_images - sr_images)
        )

        # VGG19 feature loss
        feature_loss = perceptual_loss(
            hr_images, sr_images
        )

        # Adversarial loss
        adversarial_loss = bce_loss(
            tf.ones_like(fake_logits),
            fake_logits
        )

        weighted_pixel = (
            PIXEL_WEIGHT * pixel_loss
        )

        weighted_feature = (
            PERCEPTUAL_WEIGHT * feature_loss
        )

        weighted_adversarial = (
            ADVERSARIAL_WEIGHT * adversarial_loss
        )

        g_loss = (
            weighted_pixel
            + weighted_feature
            + weighted_adversarial
        )

    tf.debugging.assert_all_finite(
        g_loss, "Non-finite generator loss"
    )

    g_gradients = g_tape.gradient(
        g_loss,
        generator.trainable_variables
    )

    g_optimizer_v2.apply_gradients(
        zip(
            g_gradients,
            generator.trainable_variables
        )
    )

    return (
        g_loss,
        d_loss,
        weighted_pixel,
        weighted_feature,
        weighted_adversarial
    )

# ------------------------------------------------------------
# 5. Validation
# ------------------------------------------------------------

@tf.function
def validate_step_v2(lr_images, hr_images):

    sr_images = generator(
        lr_images, training=False
    )

    sr_images = tf.clip_by_value(
        sr_images, 0.0, 1.0
    )

    mse = tf.reduce_mean(
        tf.square(hr_images - sr_images),
        axis=[1, 2, 3]
    )

    psnr = tf.image.psnr(
        hr_images, sr_images, max_val=1.0
    )

    ssim = tf.image.ssim(
        hr_images, sr_images, max_val=1.0
    )

    return mse, psnr, ssim

# ------------------------------------------------------------
# 6. Training history
# ------------------------------------------------------------

history_columns_v2 = [
    "epoch",
    "generator_loss",
    "discriminator_loss",
    "pixel_loss",
    "perceptual_loss",
    "adversarial_loss",
    "val_mse",
    "val_psnr",
    "val_ssim",
    "duration_seconds",
    "timestamp_utc"
]

def write_history_v2(row):

    old_rows = []

    if (
        history_path_v2.exists()
        and history_path_v2.stat().st_size > 0
    ):
        with open(history_path_v2, newline="") as f:
            old_rows = list(csv.DictReader(f))

    # Replace any previous record for this epoch
    old_rows = [
        old for old in old_rows
        if int(old["epoch"]) != row["epoch"]
    ]

    old_rows.append(row)

    old_rows.sort(
        key=lambda x: int(x["epoch"])
    )

    temporary_path = history_path_v2.with_suffix(".tmp")

    with open(temporary_path, "w", newline="") as f:

        writer = csv.DictWriter(
            f,
            fieldnames=history_columns_v2
        )

        writer.writeheader()
        writer.writerows(old_rows)

    os.replace(temporary_path, history_path_v2)

# ------------------------------------------------------------
# 7. Training loop
# ------------------------------------------------------------

print("\nSRGAN GroupNorm v2")
print("Completed epochs:", start_epoch_v2)
print("Run until epoch:", RUN_TO_EPOCH)
print("Total required epochs:", TOTAL_GAN_EPOCHS)
print("Training batches:", len(sr_train_ds))
print("Validation batches:", len(sr_val_ds))

if start_epoch_v2 >= RUN_TO_EPOCH:

    print("\nRequested epochs already completed.")

else:

    for epoch in range(start_epoch_v2, RUN_TO_EPOCH):

        epoch_start = time.perf_counter()

        metrics = {
            "g": tf.keras.metrics.Mean(),
            "d": tf.keras.metrics.Mean(),
            "pixel": tf.keras.metrics.Mean(),
            "feature": tf.keras.metrics.Mean(),
            "adv": tf.keras.metrics.Mean()
        }

        print(
            f"\nEpoch {epoch + 1}/{TOTAL_GAN_EPOCHS}"
        )

        # Training
        for step, (lr_images, hr_images) in enumerate(
            sr_train_ds, start=1
        ):

            g, d, pixel, feature, adv = train_step_v2(
                lr_images, hr_images
            )

            metrics["g"].update_state(g)
            metrics["d"].update_state(d)
            metrics["pixel"].update_state(pixel)
            metrics["feature"].update_state(feature)
            metrics["adv"].update_state(adv)

            if (
                step % 100 == 0
                or step == len(sr_train_ds)
            ):

                print(
                    f"Batch {step}/{len(sr_train_ds)} | "
                    f"G: {float(metrics['g'].result()):.5f} | "
                    f"D: {float(metrics['d'].result()):.5f}"
                )

        # Validation
        val_mse_sum = 0.0
        val_psnr_sum = 0.0
        val_ssim_sum = 0.0
        val_count = 0

        for lr_images, hr_images in sr_val_ds:

            mse, psnr, ssim = validate_step_v2(
                lr_images, hr_images
            )

            val_mse_sum += float(tf.reduce_sum(mse))
            val_psnr_sum += float(tf.reduce_sum(psnr))
            val_ssim_sum += float(tf.reduce_sum(ssim))

            val_count += int(tf.shape(lr_images)[0])

        val_mse = val_mse_sum / val_count
        val_psnr = val_psnr_sum / val_count
        val_ssim = val_ssim_sum / val_count

        elapsed = time.perf_counter() - epoch_start
        epoch_number = epoch + 1

        row = {
            "epoch": epoch_number,
            "generator_loss": float(metrics["g"].result()),
            "discriminator_loss": float(metrics["d"].result()),
            "pixel_loss": float(metrics["pixel"].result()),
            "perceptual_loss": float(metrics["feature"].result()),
            "adversarial_loss": float(metrics["adv"].result()),
            "val_mse": val_mse,
            "val_psnr": val_psnr,
            "val_ssim": val_ssim,
            "duration_seconds": round(elapsed, 2),
            "timestamp_utc": datetime.now(
                timezone.utc
            ).isoformat()
        }

        # Save the full training state
        completed_gan_epochs_v2.assign(epoch_number)

        checkpoint_path = resume_manager_v2.save(
            checkpoint_number=epoch_number
        )

        # Additional archive every 10 epochs
        if epoch_number % CHECKPOINT_INTERVAL == 0:

            archive_path = archive_manager_v2.save(
                checkpoint_number=epoch_number
            )

            print("Archive saved:", archive_path)

        write_history_v2(row)

        print("\nEpoch summary")
        print("-" * 45)
        print(
            f"Generator Loss:     "
            f"{row['generator_loss']:.6f}"
        )
        print(
            f"Discriminator Loss: "
            f"{row['discriminator_loss']:.6f}"
        )
        print(f"Validation MSE:     {val_mse:.6f}")
        print(f"Validation PSNR:    {val_psnr:.3f} dB")
        print(f"Validation SSIM:    {val_ssim:.4f}")
        print(f"Duration:           {elapsed:.1f} seconds")
        print("Checkpoint:", checkpoint_path)

        if epoch_number == 1:
            print(
                f"\nEstimated 150-epoch duration: "
                f"{elapsed * 150 / 3600:.1f} hours"
            )

        print("-" * 45)

print("\nCompleted adversarial epochs:",
      int(completed_gan_epochs_v2.numpy()))

print("Training log:", history_path_v2)
print("Latest checkpoints:", resume_dir_v2)
print("Archive checkpoints:", archive_dir_v2)


In [ ]:


import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import roc_auc_score

# Restore the latest GroupNorm v2 checkpoint
latest_v2 = resume_manager_v2.latest_checkpoint

if latest_v2 is None:
    raise FileNotFoundError("No GroupNorm v2 checkpoint found.")

checkpoint_v2.restore(
    latest_v2
).assert_existing_objects_matched()

current_epoch = int(completed_gan_epochs_v2.numpy())

# Evaluate 128 validation images
real_probabilities = []
fake_probabilities = []

max_mode_difference = 0.0

for lr_images, hr_images in sr_val_ds.take(8):

    sr_images = generator(
        lr_images, training=False
    )

    mixed_images = tf.concat(
        [hr_images, sr_images],
        axis=0
    )

    # Evaluate both classes together
    logits_eval = discriminator(
        mixed_images, training=False
    )

    logits_train = discriminator(
        mixed_images, training=True
    )

    difference = float(tf.reduce_max(
        tf.abs(logits_eval - logits_train)
    ).numpy())

    max_mode_difference = max(
        max_mode_difference, difference
    )

    batch_size = int(tf.shape(hr_images)[0])

    probabilities = tf.sigmoid(
        logits_eval
    ).numpy().flatten()

    real_probabilities.extend(
        probabilities[:batch_size]
    )

    fake_probabilities.extend(
        probabilities[batch_size:]
    )

real_probs = np.array(real_probabilities)
fake_probs = np.array(fake_probabilities)

real_accuracy = np.mean(real_probs >= 0.5)
fake_accuracy = np.mean(fake_probs < 0.5)

labels = np.concatenate([
    np.ones(len(real_probs)),
    np.zeros(len(fake_probs))
])

scores = np.concatenate([
    real_probs, fake_probs
])

discriminator_auc = roc_auc_score(
    labels, scores
)

# Display results
print(f"SRGAN v2 - Epoch {current_epoch}")
print("Discriminator diagnostics")
print("-" * 45)

print("Validation images:", len(real_probs))
print()

print(
    f"Mean probability (real): "
    f"{real_probs.mean():.4f}"
)

print(
    f"Mean probability (fake): "
    f"{fake_probs.mean():.4f}"
)

print()

print(f"Real accuracy: {real_accuracy:.2%}")
print(f"Fake accuracy: {fake_accuracy:.2%}")
print(f"Discriminator AUC: {discriminator_auc:.4f}")

print()
print(
    f"Train/inference difference: "
    f"{max_mode_difference:.8f}"
)

# Save diagnostics
diagnostic_file = (
    PROJECT_DIR / "results" /
    "srgan_diagnostics_groupnorm_v2.csv"
)

new_result = pd.DataFrame([{
    "epoch": current_epoch,
    "real_probability": real_probs.mean(),
    "fake_probability": fake_probs.mean(),
    "real_accuracy": real_accuracy,
    "fake_accuracy": fake_accuracy,
    "discriminator_auc": discriminator_auc,
    "mode_difference": max_mode_difference
}])

if diagnostic_file.exists():
    previous = pd.read_csv(diagnostic_file)
    previous = previous[
        previous["epoch"] != current_epoch
    ]
    new_result = pd.concat(
        [previous, new_result],
        ignore_index=True
    )

new_result = new_result.sort_values("epoch")
new_result.to_csv(diagnostic_file, index=False)

print("\nDiagnostics saved successfully.")
print("Checkpoint remains unchanged.")


In [ ]:

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

# --------------------------------------------------
# 1. Load saved training results
# --------------------------------------------------

history_file = (
    PROJECT_DIR / "results" /
    "srgan_training_groupnorm_v2.csv"
)

history = pd.read_csv(history_file)
history = history.sort_values("epoch")

latest_checkpoint = tf.train.latest_checkpoint(
    str(resume_dir_v2)
)

if latest_checkpoint is None:
    raise FileNotFoundError("No SRGAN v2 checkpoint found.")

# Create separate models for evaluation
eval_generator = build_generator()
eval_discriminator = build_discriminator()

eval_epoch = tf.Variable(
    0, dtype=tf.int64, trainable=False
)

eval_checkpoint = tf.train.Checkpoint(
    generator=eval_generator,
    discriminator=eval_discriminator,
    epoch=eval_epoch
)

status = eval_checkpoint.restore(latest_checkpoint)
status.expect_partial()
status.assert_existing_objects_matched()

current_epoch = int(eval_epoch.numpy())

print("Evaluating SRGAN epoch:", current_epoch)

# Load the pretrained generator
pretrained_generator = tf.keras.models.load_model(
    str(best_generator_path),
    compile=False
)

# --------------------------------------------------
# 2. Compare reconstructed images
# --------------------------------------------------

examples = pd.concat([
    val_df[val_df["label"] == 0].sample(
        n=2, random_state=SEED
    ),
    val_df[val_df["label"] == 1].sample(
        n=2, random_state=SEED
    )
]).reset_index(drop=True)

fig, axes = plt.subplots(
    4, 5, figsize=(16, 12)
)

titles = [
    "LR (32x32)",
    "Bicubic",
    "Pretrained Generator",
    f"SRGAN Epoch {current_epoch}",
    "Original HR"
]

for i, row in examples.iterrows():

    lr, hr = load_sr_pair(
        tf.constant(row["prepared_path"]),
        training=False
    )

    input_batch = tf.expand_dims(lr, 0)

    bicubic = tf.image.resize(
        lr,
        [HR_SIZE, HR_SIZE],
        method="bicubic"
    )

    pretrained_sr = pretrained_generator(
        input_batch, training=False
    )[0]

    gan_sr = eval_generator(
        input_batch, training=False
    )[0]

    images = [
        lr, bicubic, pretrained_sr, gan_sr, hr
    ]

    for j, image in enumerate(images):

        image = tf.clip_by_value(
            image, 0.0, 1.0
        )

        axes[i, j].imshow(
            image.numpy(),
            interpolation="nearest" if j == 0 else "none"
        )

        axes[i, j].axis("off")

        if i == 0:
            axes[i, j].set_title(
                titles[j], fontsize=11
            )

    label = "Cat" if row["label"] == 0 else "Dog"

    axes[i, 0].set_ylabel(
        label,
        fontweight="bold"
    )

plt.suptitle(
    f"SRGAN Image Comparison - Epoch {current_epoch}",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout()

figure_path = (
    PROJECT_DIR / "figures" /
    f"srgan_v2_epoch_{current_epoch:03d}_comparison.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Image comparison saved:", figure_path)

# --------------------------------------------------
# 3. Training progress curves
# --------------------------------------------------

fig, axes = plt.subplots(
    2, 2, figsize=(12, 9)
)

plots = [
    ("generator_loss", "Generator Loss"),
    ("discriminator_loss", "Discriminator Loss"),
    ("val_psnr", "Validation PSNR (dB)"),
    ("val_ssim", "Validation SSIM")
]

for ax, (column, title) in zip(
    axes.flatten(), plots
):

    ax.plot(
        history["epoch"],
        history[column],
        marker="o",
        linewidth=2
    )

    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.3)

axes[1, 0].axhline(
    25.689,
    linestyle="--",
    color="gray",
    label="Pretrained Generator"
)

axes[1, 1].axhline(
    0.7436,
    linestyle="--",
    color="gray",
    label="Pretrained Generator"
)

axes[1, 0].legend()
axes[1, 1].legend()

plt.suptitle(
    "SRGAN v2 - Training Progress",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout()

curves_path = (
    PROJECT_DIR / "figures" /
    f"srgan_v2_training_epoch_{current_epoch:03d}.png"
)

plt.savefig(
    curves_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Training curves saved:", curves_path)

# --------------------------------------------------
# 4. Discriminator evaluation
# --------------------------------------------------

real_probs = []
fake_probs = []

for lr_images, hr_images in sr_val_ds.take(8):

    sr_images = eval_generator(
        lr_images, training=False
    )

    combined = tf.concat(
        [hr_images, sr_images],
        axis=0
    )

    scores = tf.sigmoid(
        eval_discriminator(
            combined, training=False
        )
    ).numpy().flatten()

    n = int(tf.shape(hr_images)[0])

    real_probs.extend(scores[:n])
    fake_probs.extend(scores[n:])

real_probs = np.array(real_probs)
fake_probs = np.array(fake_probs)

labels = np.concatenate([
    np.ones(len(real_probs)),
    np.zeros(len(fake_probs))
])

scores = np.concatenate([
    real_probs, fake_probs
])

auc = roc_auc_score(labels, scores)

print("\nDiscriminator diagnostics")
print("-" * 45)

print(
    f"Real probability: {real_probs.mean():.4f}"
)
print(
    f"Fake probability: {fake_probs.mean():.4f}"
)

print(
    f"Real accuracy: {(real_probs >= 0.5).mean():.2%}"
)
print(
    f"Fake accuracy: {(fake_probs < 0.5).mean():.2%}"
)

print(f"Discriminator AUC: {auc:.4f}")

# Save diagnostics
diagnostic_file = (
    PROJECT_DIR / "results" /
    "srgan_diagnostics_groupnorm_v2.csv"
)

new_diagnostic = pd.DataFrame([{
    "epoch": current_epoch,
    "real_probability": float(real_probs.mean()),
    "fake_probability": float(fake_probs.mean()),
    "real_accuracy": float((real_probs >= 0.5).mean()),
    "fake_accuracy": float((fake_probs < 0.5).mean()),
    "discriminator_auc": float(auc),
    "mode_difference": 0.0
}])

if diagnostic_file.exists():
    old = pd.read_csv(diagnostic_file)
    old = old[old["epoch"] != current_epoch]
    new_diagnostic = pd.concat(
        [old, new_diagnostic],
        ignore_index=True
    )

new_diagnostic.sort_values(
    "epoch"
).to_csv(diagnostic_file, index=False)

print("\nEvaluation completed.")
print("Original training checkpoints unchanged.")


In [ ]:
import os
import csv
import json
import time
import tensorflow as tf
from pathlib import Path
from datetime import datetime, timezone

# ============================================================
# SRGAN TRAINING V2 + BEST GENERATOR SAVING
# ============================================================

TOTAL_GAN_EPOCHS = 150
RUN_TO_EPOCH = 150
CHECKPOINT_INTERVAL = 10

assert discriminator.name == "SRGAN_Discriminator_GroupNorm", (
    "Run Cell 20 first."
)

feature_extractor.trainable = False

# ------------------------------------------------------------
# 1. Directories
# ------------------------------------------------------------

gan_v2_dir = (
    PROJECT_DIR / "checkpoints" /
    "srgan" / "groupnorm_v2"
)

resume_dir_v2 = gan_v2_dir / "resume"
archive_dir_v2 = gan_v2_dir / "archive"
best_dir_v2 = gan_v2_dir / "best_generators"

resume_dir_v2.mkdir(parents=True, exist_ok=True)
archive_dir_v2.mkdir(parents=True, exist_ok=True)
best_dir_v2.mkdir(parents=True, exist_ok=True)

history_path_v2 = (
    PROJECT_DIR / "results" /
    "srgan_training_groupnorm_v2.csv"
)

best_psnr_model_path = best_dir_v2 / "generator_best_psnr.keras"
best_ssim_model_path = best_dir_v2 / "generator_best_ssim.keras"
tracking_path = best_dir_v2 / "best_generator_tracking.json"

# ------------------------------------------------------------
# 2. Optimizers
# ------------------------------------------------------------

g_optimizer_v2 = tf.keras.optimizers.Adam(
    learning_rate=GENERATOR_LR,
    beta_1=0.9,
    global_clipnorm=1.0
)

d_optimizer_v2 = tf.keras.optimizers.Adam(
    learning_rate=DISCRIMINATOR_LR,
    beta_1=0.9,
    global_clipnorm=1.0
)

g_optimizer_v2.build(generator.trainable_variables)
d_optimizer_v2.build(discriminator.trainable_variables)

completed_gan_epochs_v2 = tf.Variable(
    0, dtype=tf.int64, trainable=False
)

# ------------------------------------------------------------
# 3. Full checkpoint restore
# ------------------------------------------------------------

checkpoint_v2 = tf.train.Checkpoint(
    generator=generator,
    discriminator=discriminator,
    generator_optimizer=g_optimizer_v2,
    discriminator_optimizer=d_optimizer_v2,
    epoch=completed_gan_epochs_v2
)

resume_manager_v2 = tf.train.CheckpointManager(
    checkpoint_v2,
    directory=str(resume_dir_v2),
    max_to_keep=3
)

archive_manager_v2 = tf.train.CheckpointManager(
    checkpoint_v2,
    directory=str(archive_dir_v2),
    max_to_keep=None
)

if resume_manager_v2.latest_checkpoint:
    checkpoint_v2.restore(
        resume_manager_v2.latest_checkpoint
    ).assert_existing_objects_matched()

    print("Restored checkpoint:")
    print(resume_manager_v2.latest_checkpoint)
else:
    raise FileNotFoundError(
        "No previous GroupNorm v2 checkpoint found."
    )

start_epoch_v2 = int(completed_gan_epochs_v2.numpy())

if start_epoch_v2 < 1:
    raise ValueError("Expected an already trained SRGAN v2 state.")

# ------------------------------------------------------------
# 4. Load history and initialize best-saving
# ------------------------------------------------------------

if not history_path_v2.exists():
    raise FileNotFoundError(
        "Training history file missing."
    )

history_df = tf.keras.utils.get_file  # dummy line removed below
import pandas as pd
history_df = pd.read_csv(history_path_v2).sort_values("epoch")

current_row = history_df[
    history_df["epoch"] == start_epoch_v2
]

if current_row.empty:
    raise ValueError(
        f"No history row found for epoch {start_epoch_v2}."
    )

current_row = current_row.iloc[0]

current_psnr = float(current_row["val_psnr"])
current_ssim = float(current_row["val_ssim"])

# Export current generator snapshot (epoch 15 now)
current_export_path = (
    best_dir_v2 / f"generator_epoch_{start_epoch_v2:03d}.keras"
)

if not current_export_path.exists():
    generator.save(str(current_export_path))
    print(f"Saved current generator snapshot: {current_export_path}")

# Initialize tracking
if tracking_path.exists():
    with open(tracking_path, "r") as f:
        tracking = json.load(f)

    best_psnr_value = float(tracking["best_psnr"]["value"])
    best_ssim_value = float(tracking["best_ssim"]["value"])

else:
    # Start best tracking from current epoch 15
    generator.save(str(best_psnr_model_path))
    generator.save(str(best_ssim_model_path))

    tracking = {
        "best_psnr": {
            "epoch": start_epoch_v2,
            "value": current_psnr,
            "path": str(best_psnr_model_path)
        },
        "best_ssim": {
            "epoch": start_epoch_v2,
            "value": current_ssim,
            "path": str(best_ssim_model_path)
        },
        "latest_exported_epoch": start_epoch_v2
    }

    with open(tracking_path, "w") as f:
        json.dump(tracking, f, indent=4)

    best_psnr_value = current_psnr
    best_ssim_value = current_ssim

    print("Initialized best-generator tracking from current epoch.")

print(f"Current exported baseline epoch: {start_epoch_v2}")
print(f"Baseline PSNR: {current_psnr:.3f} dB")
print(f"Baseline SSIM: {current_ssim:.4f}")
print(f"Tracked best PSNR: {best_psnr_value:.3f} dB")
print(f"Tracked best SSIM: {best_ssim_value:.4f}")

# ------------------------------------------------------------
# 5. Train step
# ------------------------------------------------------------

@tf.function
def train_step_v2(lr_images, hr_images):

    # ----- Discriminator -----
    generated_for_d = tf.stop_gradient(
        generator(lr_images, training=False)
    )

    batch_size = tf.shape(hr_images)[0]

    combined_images = tf.concat(
        [hr_images, generated_for_d],
        axis=0
    )

    combined_labels = tf.concat([
        tf.ones((batch_size, 1)),
        tf.zeros((batch_size, 1))
    ], axis=0)

    indices = tf.random.shuffle(
        tf.range(tf.shape(combined_images)[0])
    )

    combined_images = tf.gather(
        combined_images, indices
    )

    combined_labels = tf.gather(
        combined_labels, indices
    )

    with tf.GradientTape() as d_tape:

        mixed_logits = discriminator(
            combined_images,
            training=True
        )

        d_loss = bce_loss(
            combined_labels,
            mixed_logits
        )

    tf.debugging.assert_all_finite(
        d_loss, "Non-finite discriminator loss"
    )

    d_grads = d_tape.gradient(
        d_loss,
        discriminator.trainable_variables
    )

    d_optimizer_v2.apply_gradients(
        zip(d_grads, discriminator.trainable_variables)
    )

    # ----- Generator -----
    with tf.GradientTape() as g_tape:

        sr_images = generator(
            lr_images, training=True
        )

        fake_logits = discriminator(
            sr_images, training=False
        )

        pixel_loss = tf.reduce_mean(
            tf.square(hr_images - sr_images)
        )

        feature_loss = perceptual_loss(
            hr_images, sr_images
        )

        adversarial_loss = bce_loss(
            tf.ones_like(fake_logits),
            fake_logits
        )

        weighted_pixel = PIXEL_WEIGHT * pixel_loss
        weighted_feature = PERCEPTUAL_WEIGHT * feature_loss
        weighted_adversarial = ADVERSARIAL_WEIGHT * adversarial_loss

        g_loss = (
            weighted_pixel
            + weighted_feature
            + weighted_adversarial
        )

    tf.debugging.assert_all_finite(
        g_loss, "Non-finite generator loss"
    )

    g_grads = g_tape.gradient(
        g_loss,
        generator.trainable_variables
    )

    g_optimizer_v2.apply_gradients(
        zip(g_grads, generator.trainable_variables)
    )

    return (
        g_loss,
        d_loss,
        weighted_pixel,
        weighted_feature,
        weighted_adversarial
    )

# ------------------------------------------------------------
# 6. Validation step
# ------------------------------------------------------------

@tf.function
def validate_step_v2(lr_images, hr_images):

    sr_images = generator(
        lr_images, training=False
    )

    sr_images = tf.clip_by_value(
        sr_images, 0.0, 1.0
    )

    mse = tf.reduce_mean(
        tf.square(hr_images - sr_images),
        axis=[1, 2, 3]
    )

    psnr = tf.image.psnr(
        hr_images, sr_images, max_val=1.0
    )

    ssim = tf.image.ssim(
        hr_images, sr_images, max_val=1.0
    )

    return mse, psnr, ssim

# ------------------------------------------------------------
# 7. History writing
# ------------------------------------------------------------

history_columns_v2 = [
    "epoch",
    "generator_loss",
    "discriminator_loss",
    "pixel_loss",
    "perceptual_loss",
    "adversarial_loss",
    "val_mse",
    "val_psnr",
    "val_ssim",
    "duration_seconds",
    "timestamp_utc"
]

def write_history_v2(row):

    old_rows = []

    if (
        history_path_v2.exists()
        and history_path_v2.stat().st_size > 0
    ):
        with open(history_path_v2, newline="") as f:
            old_rows = list(csv.DictReader(f))

    old_rows = [
        old for old in old_rows
        if int(old["epoch"]) != row["epoch"]
    ]

    old_rows.append(row)
    old_rows.sort(key=lambda x: int(x["epoch"]))

    tmp_path = history_path_v2.with_suffix(".tmp")

    with open(tmp_path, "w", newline="") as f:
        writer = csv.DictWriter(
            f, fieldnames=history_columns_v2
        )
        writer.writeheader()
        writer.writerows(old_rows)

    os.replace(tmp_path, history_path_v2)

# ------------------------------------------------------------
# 8. Training loop
# ------------------------------------------------------------

print("\nSRGAN GroupNorm v2 + best saving")
print("Completed epochs:", start_epoch_v2)
print("Run until epoch:", RUN_TO_EPOCH)
print("Training batches:", len(sr_train_ds))
print("Validation batches:", len(sr_val_ds))

if start_epoch_v2 >= RUN_TO_EPOCH:

    print("\nRequested epochs already completed.")

else:

    for epoch in range(start_epoch_v2, RUN_TO_EPOCH):

        epoch_start = time.perf_counter()

        metrics = {
            "g": tf.keras.metrics.Mean(),
            "d": tf.keras.metrics.Mean(),
            "pixel": tf.keras.metrics.Mean(),
            "feature": tf.keras.metrics.Mean(),
            "adv": tf.keras.metrics.Mean()
        }

        print(f"\nEpoch {epoch + 1}/{TOTAL_GAN_EPOCHS}")

        for step, (lr_images, hr_images) in enumerate(
            sr_train_ds, start=1
        ):

            g, d, pixel, feature, adv = train_step_v2(
                lr_images, hr_images
            )

            metrics["g"].update_state(g)
            metrics["d"].update_state(d)
            metrics["pixel"].update_state(pixel)
            metrics["feature"].update_state(feature)
            metrics["adv"].update_state(adv)

            if step % 100 == 0 or step == len(sr_train_ds):
                print(
                    f"Batch {step}/{len(sr_train_ds)} | "
                    f"G: {float(metrics['g'].result()):.5f} | "
                    f"D: {float(metrics['d'].result()):.5f}"
                )

        # ----- Validation -----
        val_mse_sum = 0.0
        val_psnr_sum = 0.0
        val_ssim_sum = 0.0
        val_count = 0

        for lr_images, hr_images in sr_val_ds:

            mse, psnr, ssim = validate_step_v2(
                lr_images, hr_images
            )

            val_mse_sum += float(tf.reduce_sum(mse))
            val_psnr_sum += float(tf.reduce_sum(psnr))
            val_ssim_sum += float(tf.reduce_sum(ssim))
            val_count += int(tf.shape(lr_images)[0])

        val_mse = val_mse_sum / val_count
        val_psnr = val_psnr_sum / val_count
        val_ssim = val_ssim_sum / val_count

        elapsed = time.perf_counter() - epoch_start
        epoch_number = epoch + 1

        row = {
            "epoch": epoch_number,
            "generator_loss": float(metrics["g"].result()),
            "discriminator_loss": float(metrics["d"].result()),
            "pixel_loss": float(metrics["pixel"].result()),
            "perceptual_loss": float(metrics["feature"].result()),
            "adversarial_loss": float(metrics["adv"].result()),
            "val_mse": val_mse,
            "val_psnr": val_psnr,
            "val_ssim": val_ssim,
            "duration_seconds": round(elapsed, 2),
            "timestamp_utc": datetime.now(
                timezone.utc
            ).isoformat()
        }

        completed_gan_epochs_v2.assign(epoch_number)

        checkpoint_path = resume_manager_v2.save(
            checkpoint_number=epoch_number
        )

        if epoch_number % CHECKPOINT_INTERVAL == 0:
            archive_path = archive_manager_v2.save(
                checkpoint_number=epoch_number
            )
            print("Archive saved:", archive_path)

        write_history_v2(row)

        # ----- Save best generators -----
        psnr_saved = False
        ssim_saved = False

        if val_psnr > best_psnr_value:
            best_psnr_value = val_psnr
            generator.save(str(best_psnr_model_path))
            tracking["best_psnr"] = {
                "epoch": epoch_number,
                "value": float(val_psnr),
                "path": str(best_psnr_model_path)
            }
            psnr_saved = True

        if val_ssim > best_ssim_value:
            best_ssim_value = val_ssim
            generator.save(str(best_ssim_model_path))
            tracking["best_ssim"] = {
                "epoch": epoch_number,
                "value": float(val_ssim),
                "path": str(best_ssim_model_path)
            }
            ssim_saved = True

        tracking["latest_exported_epoch"] = epoch_number

        with open(tracking_path, "w") as f:
            json.dump(tracking, f, indent=4)

        print("\nEpoch summary")
        print("-" * 50)
        print(f"Generator Loss:     {row['generator_loss']:.6f}")
        print(f"Discriminator Loss: {row['discriminator_loss']:.6f}")
        print(f"Validation MSE:     {val_mse:.6f}")
        print(f"Validation PSNR:    {val_psnr:.3f} dB")
        print(f"Validation SSIM:    {val_ssim:.4f}")
        print(f"Duration:           {elapsed:.1f} seconds")
        print("Checkpoint:", checkpoint_path)

        if psnr_saved:
            print(
                f"New best PSNR model saved "
                f"(epoch {epoch_number})"
            )

        if ssim_saved:
            print(
                f"New best SSIM model saved "
                f"(epoch {epoch_number})"
            )

        print("-" * 50)

print("\nCompleted adversarial epochs:",
      int(completed_gan_epochs_v2.numpy()))

print("History file:", history_path_v2)
print("Resume checkpoints:", resume_dir_v2)
print("Archive checkpoints:", archive_dir_v2)
print("Best generator tracking:", tracking_path)
print("Best PSNR model:", best_psnr_model_path)
print("Best SSIM model:", best_ssim_model_path)

In [ ]:

import tensorflow as tf
import json
import pandas as pd
from pathlib import Path
from google.colab import drive

# Reconnect Google Drive
drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Applied_AI_Midterm_SRGAN"
)

# Check saved training state
gan_dir = (
    PROJECT_DIR / "checkpoints" /
    "srgan" / "groupnorm_v2"
)

resume_dir = gan_dir / "resume"
archive_dir = gan_dir / "archive"
best_dir = gan_dir / "best_generators"

latest = tf.train.latest_checkpoint(str(resume_dir))
archive = tf.train.latest_checkpoint(str(archive_dir))

print("\n=== SRGAN RECOVERY CHECK ===")

print("GPU:", tf.config.list_physical_devices("GPU"))
print("Latest checkpoint:", latest)
print("Latest archive:", archive)

if latest:
    print(
        "Checkpoint index exists:",
        Path(latest + ".index").exists()
    )
else:
    print("WARNING: No resume checkpoint found!")

# Check training history
history_file = (
    PROJECT_DIR / "results" /
    "srgan_training_groupnorm_v2.csv"
)

if history_file.exists():
    history = pd.read_csv(history_file)
    if not history.empty:
        print(
            "\nLast recorded epoch:",
            int(history["epoch"].max())
        )
        print("\nLast training result:")
        print(history.tail(1).to_string(index=False))

# Check best saved generators
tracking_file = best_dir / "best_generator_tracking.json"

if tracking_file.exists():
    with open(tracking_file) as f:
        tracking = json.load(f)

    print("\nBest PSNR model:")
    print(tracking["best_psnr"])

    print("\nBest SSIM model:")
    print(tracking["best_ssim"])

print("\nRecovery check completed.")


In [ ]:

import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

if tf.config.list_physical_devices("GPU"):
    print("\nSUCCESS: GPU ready!")
else:
    print("\nWARNING: No GPU detected!")



In [ ]:

import tensorflow as tf
import zipfile
import hashlib
import json
from pathlib import Path
from google.colab import drive, files

# Connect Google Drive
drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Applied_AI_Midterm_SRGAN"
)

# Locate latest SRGAN checkpoint
resume_dir = (
    PROJECT_DIR / "checkpoints" /
    "srgan/groupnorm_v2/resume"
)

latest_checkpoint = tf.train.latest_checkpoint(
    str(resume_dir)
)

if latest_checkpoint is None:
    raise FileNotFoundError(
        "No SRGAN checkpoint found!"
    )

checkpoint_name = Path(latest_checkpoint).name

# Collect checkpoint files
checkpoint_files = sorted([
    Path(str(latest_checkpoint) + ".index"),
    *resume_dir.glob(checkpoint_name + ".data-*")
])

if (
    not checkpoint_files
    or not all(p.is_file() for p in checkpoint_files)
    or len(checkpoint_files) < 2
):
    raise RuntimeError(
        "Checkpoint is incomplete. Export cancelled."
    )

# Additional files for recovery and reproducibility
required_files = [
    "data/dataset_split.csv",
    "results/srgan_training_groupnorm_v2.csv",
    "checkpoints/srgan/pretraining/generator_pretrain_best.keras",
    "checkpoints/model_A/model_A_final.keras",
    "documentation/srgan_architecture_v2.json",
    "documentation/srgan_training_config.json"
]

optional_files = [
    "data/excluded_images.csv",
    "documentation/dataset_info.json",
    "results/generator_pretraining.csv",
    "results/generator_pretraining_metrics.csv",
    "results/model_A_training.csv",
    "results/model_A_finetuning.csv",
    "results/srgan_diagnostics_groupnorm_v2.csv"
]

missing = [
    item for item in required_files
    if not (PROJECT_DIR / item).is_file()
]

if missing:
    raise FileNotFoundError(
        "Missing required files: " + str(missing)
    )

# Include all exported best-generator files
best_dir = (
    PROJECT_DIR / "checkpoints" /
    "srgan/groupnorm_v2/best_generators"
)

best_files = sorted(
    p for p in best_dir.rglob("*") if p.is_file()
)

if not best_files:
    raise FileNotFoundError(
        "Best-generator files not found."
    )

# Create transfer ZIP
zip_path = Path("/content/srgan_kaggle_transfer.zip")

with zipfile.ZipFile(
    zip_path, "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=1
) as zf:

    # Latest training checkpoint
    for path in checkpoint_files:
        relative = path.relative_to(PROJECT_DIR)
        zf.write(path, "project/" + str(relative))

    # Create checkpoint pointer for the exported epoch
    zf.writestr(
        "project/checkpoints/srgan/groupnorm_v2/resume/checkpoint",
        (
            f'model_checkpoint_path: "{checkpoint_name}"\n'
            f'all_model_checkpoint_paths: "{checkpoint_name}"\n'
        )
    )

    # Required and optional files
    for relative in required_files + optional_files:
        path = PROJECT_DIR / relative

        if path.is_file():
            zf.write(path, "project/" + relative)

    # Best-generator models and tracking metadata
    for path in best_files:
        relative = path.relative_to(PROJECT_DIR)
        zf.write(path, "project/" + str(relative))

    # Transfer manifest
    manifest = {
        "source": "Google Colab",
        "destination": "Kaggle Notebooks",
        "checkpoint": checkpoint_name,
        "tensorflow_source_version": tf.__version__,
        "dataset": "blourdhuraju/dogs-vs-cats-classification",
        "dataset_version": 1
    }

    zf.writestr(
        "project/kaggle_transfer_manifest.json",
        json.dumps(manifest, indent=4)
    )

# Verify ZIP integrity
with zipfile.ZipFile(zip_path) as zf:
    bad_file = zf.testzip()

if bad_file is not None:
    raise RuntimeError(
        f"ZIP integrity check failed: {bad_file}"
    )

# Calculate checksum
sha256 = hashlib.sha256()

with open(zip_path, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

size_mb = zip_path.stat().st_size / (1024 ** 2)

print("\n=== KAGGLE TRANSFER PACKAGE ===")
print("Checkpoint:", checkpoint_name)
print(f"ZIP size: {size_mb:.1f} MB")
print("SHA256:", sha256.hexdigest())
print("ZIP integrity: PASSED")
print("File:", zip_path)

print("\nReady to download!")

# Download to your computer
files.download(str(zip_path))

